
# Ethereum Accumulation Zones
## The Overfitters – Machine Learning Project

### Forskningsfråga
Kan logaritmisk regression identifiera historiskt
gynnsamma köpzoner för Ethereum med hjälp av
ETH- och BTC-pristidsserier?

### Syfte
Att undersöka om köp av Ethereum i historiskt
låga regressionszoner har genererat högre
framtida avkastning än köp i andra priszoner.

### Metod
1. Historisk data från 2015
2. Explorativ dataanalys (EDA)
3. Logaritmisk regression
4. Identifiering av ackumulationszoner
5. Utvärdering efter 6 och 12 månader
6. Expanding Window Cross-Validation
7. Backtesting mot Buy & Hold
8. 12-månadersprojektion
9. Visualisering i Streamlit

### Metodologiska begränsningar
Regressionsmodellen anpassas endast på historisk
information som var tillgänglig vid varje tidpunkt.

Testdata hålls separat från modellutvecklingen.

Historiska samband innebär inte att framtida
avkastning kan förutsägas med säkerhet.


In [1]:

# STEG 1 – Läs in historiska data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("data")

# Läs prisdata
df = pd.read_csv(
    DATA_DIR / "crypto_historical_prices.csv"
)

# Hantera datum
df["time"] = pd.to_datetime(
    df["time"],
    utc=True
)

df = df.sort_values(
    "time"
).reset_index(drop=True)

# Säkerställ numeriska priser
for col in ["ETH_Price", "BTC_Price"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )
    df[col] = df[col].where(df[col] > 0)

# Separata dataset
df_eth = df[
    ["time", "ETH_Price"]
].dropna().copy()

df_btc = df[
    ["time", "BTC_Price"]
].dropna().copy()

print("ETH observationer:", len(df_eth))
print("BTC observationer:", len(df_btc))

print("\nEthereum:")
print("Första datum:", df_eth["time"].min())
print("Sista datum:", df_eth["time"].max())

print("\nBitcoin:")
print("Första datum:", df_btc["time"].min())
print("Sista datum:", df_btc["time"].max())

print("\nSenaste priser:")
display(df.tail(5))


ETH observationer: 4104
BTC observationer: 4323

Ethereum:
Första datum: 2015-08-08 00:00:00+00:00
Sista datum: 2026-11-01 00:00:00+00:00

Bitcoin:
Första datum: 2015-01-01 00:00:00+00:00
Sista datum: 2026-11-01 00:00:00+00:00

Senaste priser:


,time,BTC_Price,ETH_Price
4318,2026-10-28 00:00:00+00:00,81237.437826,2356.988509
4319,2026-10-29 00:00:00+00:00,78193.426404,2256.031994
4320,2026-10-30 00:00:00+00:00,77109.590747,2240.229980
4321,2026-10-31 00:00:00+00:00,77401.017287,2252.073358
4322,2026-11-01 00:00:00+00:00,78135.080289,2269.988406



### Resultat och slutsats – Steg 1

Vi har importerat historiska dagspriser för Ethereum (ETH)
och Bitcoin (BTC) från 2015 till november 2026.

**Resultat:**
- Ethereum: 4 104 observationer, 2015-08-08 till 2026-11-01.
- Bitcoin: 4 323 observationer, 2015-01-01 till 2026-11-01.
- Ethereum saknar prisobservationer före augusti 2015.
- Båda dataserierna har samma sista observationsdatum.

**Slutsats:**

Datasetet innehåller tillräckligt lång historik för att
undersöka flera marknadscykler. Ethereum används som
huvudsakligt analysobjekt medan Bitcoin används som
jämförelsevariabel.

Historiska priser är inte samma sak som livekurser.
```

## Steg 2 – Explorativ dataanalys (EDA)

<CodeBlock language="markdown" editable>
### Resultat och slutsats – Steg 2

Vi har undersökt den historiska prisutvecklingen
för Ethereum och Bitcoin med interaktiva diagram.

**Metoder:**
- Linjär prisskala.
- Logaritmisk prisskala.
- Beskrivande statistik.
- Normaliserad jämförelse mellan ETH och BTC.

**Resultat:**

Både Ethereum och Bitcoin uppvisar stora historiska
prisvariationer och perioder med kraftiga uppgångar
och nedgångar.

Den logaritmiska skalan gör det lättare att jämföra
procentuella prisförändringar över långa tidsperioder.

Genom att normalisera priserna till index 100 kan
vi jämföra valutornas relativa prisutveckling,
trots att deras nominella priser skiljer sig.

**Slutsats:**

EDA visar att kryptomarknaden kännetecknas av
kraftiga prisrörelser och förändrade marknadsförhållanden.

Det motiverar användningen av logaritmiska modeller
och tidsbaserad validering.



## Steg 2 – Explorativ dataanalys (EDA)

Vi undersöker historisk prisutveckling för Ethereum
och Bitcoin från 2015 till 2026.

Syftet är att förstå prisernas utveckling innan
vi bygger en logaritmisk regressionsmodell.

Vi analyserar:

- Historiska prisrörelser
- Beskrivande statistik
- Linjär och logaritmisk prisskala
- Sambandet mellan ETH och BTC

Ethereum är vårt huvudsakliga analysobjekt.
Bitcoin används som jämförelse och möjlig
förklarande variabel.

EDA används för att förstå historiska mönster,
inte för att bevisa framtida prognosförmåga.


In [2]:
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import nbformat
import plotly.io as pio

print("nbformat version:", nbformat.__version__)

pio.renderers.default = "plotly_mimetype"

nbformat version: 5.11.1


In [4]:

# STEG 2 – Interaktiv EDA för Ethereum och Bitcoin

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

# Separata dataset
assets = {
    "Ethereum (ETH)": ("ETH_Price", df_eth),
    "Bitcoin (BTC)": ("BTC_Price", df_btc)
}

statistics = []

for name, (price_col, data) in assets.items():

    data = data.sort_values("time").copy()
    prices = data[price_col]

    # Statistik
    statistics.append({
        "Kryptovaluta": name,
        "Observationer": len(data),
        "Medelpris USD": prices.mean(),
        "Median USD": prices.median(),
        "Standardavvikelse": prices.std(),
        "Min USD": prices.min(),
        "Max USD": prices.max()
    })

    # 1. Linjärt prisdiagram
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data[price_col],
        mode="lines",
        name=name
    ))

    fig.update_layout(
        title=f"{name} – Historisk prisutveckling",
        xaxis_title="Datum",
        yaxis_title="Pris (USD)",
        template="plotly_dark",
        height=500,
        hovermode="x unified"
    )

    fig.update_xaxes(
        rangeslider_visible=True
    )

    fig.show()

    # 2. Logaritmiskt prisdiagram
    fig_log = go.Figure()

    fig_log.add_trace(go.Scatter(
        x=data["time"],
        y=data[price_col],
        mode="lines",
        name=name
    ))

    fig_log.update_layout(
        title=f"{name} – Logaritmisk prisutveckling",
        xaxis_title="Datum",
        yaxis_title="Pris (USD, logaritmisk skala)",
        yaxis_type="log",
        template="plotly_dark",
        height=500,
        hovermode="x unified"
    )

    fig_log.update_xaxes(
        rangeslider_visible=True
    )

    fig_log.show()

# Statistiktabell
df_statistics = pd.DataFrame(statistics)

print("Beskrivande statistik:")
display(df_statistics.round(2))


Beskrivande statistik:


,Kryptovaluta,Observationer,Medelpris USD,Median USD,Standardavvikelse,Min USD,Max USD
0,Ethereum (ETH),4104,1400.68,1236.95,1303.55,0.42,4831.10
1,Bitcoin (BTC),4323,30818.66,16696.00,32951.43,175.64,124824.45



## Steg 3 – Logaritmisk regression och ackumulationszoner

### Syfte

Vi undersöker om en logaritmisk regressionsmodell
kan beskriva Ethereums långsiktiga prisutveckling.

Modellen anpassas till logaritmen av ETH-priset
som funktion av tiden.

Vi skapar tre zoner:

- Nedre zon: ETH-priset ligger under det nedre regressionsbandet.
- Neutral zon: ETH-priset ligger mellan regressionsbanden.
- Övre zon: ETH-priset ligger över det övre regressionsbandet.

Bandens bredd baseras på standardavvikelsen
för modellens residualer.

Den första modellen används enbart för EDA.
Den får inte användas för historisk backtesting,
eftersom den tränas på hela tidsperioden.

Senare använder vi Expanding Window Validation
för att undvika framtidsinformation.


In [5]:

# STEG 3 – Logaritmisk regression för Ethereum
# Beskrivande modell – INTE för backtesting

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.linear_model import LinearRegression

# Kopiera Ethereum-data
df_reg = df_eth.copy()

df_reg = df_reg.sort_values(
    "time"
).reset_index(drop=True)

# Säkerställ giltiga priser
df_reg = df_reg[
    df_reg["ETH_Price"] > 0
].copy()

# Antal dagar sedan första observationen
df_reg["Days"] = (
    df_reg["time"] - df_reg["time"].min()
).dt.days

# Logaritmera priset
df_reg["Log_Price"] = np.log(
    df_reg["ETH_Price"]
)

# Träna beskrivande regressionsmodell
X = df_reg[["Days"]]
y = df_reg["Log_Price"]

reg_model = LinearRegression()
reg_model.fit(X, y)

# Beräkna logaritmisk trend
df_reg["Predicted_Log"] = reg_model.predict(X)

# Residualer
df_reg["Residual"] = (
    df_reg["Log_Price"]
    - df_reg["Predicted_Log"]
)

# Residualernas standardavvikelse
residual_std = df_reg["Residual"].std(ddof=1)

# Regressionsband: +/- 1 standardavvikelse
df_reg["Regression_Trend"] = np.exp(
    df_reg["Predicted_Log"]
)

df_reg["Upper_Band"] = np.exp(
    df_reg["Predicted_Log"] + residual_std
)

df_reg["Lower_Band"] = np.exp(
    df_reg["Predicted_Log"] - residual_std
)

# Klassificera historiska zoner
df_reg["Price_Zone"] = np.select(
    [
        df_reg["ETH_Price"] < df_reg["Lower_Band"],
        df_reg["ETH_Price"] > df_reg["Upper_Band"]
    ],
    [
        "Nedre zon",
        "Övre zon"
    ],
    default="Neutral zon"
)

# Visa statistik
print("Antal observationer:", len(df_reg))
print("Residual standardavvikelse:", round(residual_std, 4))
print("Regression R²:", round(reg_model.score(X, y), 4))

print("\nAntal dagar per zon:")
display(
    df_reg["Price_Zone"]
    .value_counts()
    .rename_axis("Zon")
    .reset_index(name="Antal dagar")
)

# Interaktivt diagram
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_reg["time"],
    y=df_reg["Upper_Band"],
    name="Övre regressionsband",
    mode="lines",
    line=dict(color="#2ECC71", width=1)
))

fig.add_trace(go.Scatter(
    x=df_reg["time"],
    y=df_reg["Lower_Band"],
    name="Nedre regressionsband",
    mode="lines",
    fill="tonexty",
    fillcolor="rgba(46,204,113,0.16)",
    line=dict(color="#2ECC71", width=1)
))

fig.add_trace(go.Scatter(
    x=df_reg["time"],
    y=df_reg["Regression_Trend"],
    name="Logaritmisk regression",
    mode="lines",
    line=dict(color="#F39C12", width=2)
))

fig.add_trace(go.Scatter(
    x=df_reg["time"],
    y=df_reg["ETH_Price"],
    name="Ethereum – pris",
    mode="lines",
    line=dict(color="#5DADE2", width=1.5)
))

fig.update_layout(
    title="Ethereum – Logaritmisk regression och ackumulationszoner",
    xaxis_title="Datum",
    yaxis_title="ETH-pris (USD, logaritmisk skala)",
    yaxis_type="log",
    template="plotly_dark",
    hovermode="x unified",
    height=650
)

fig.update_xaxes(rangeslider_visible=True)

fig.show()


Antal observationer: 4104
Residual standardavvikelse: 1.203
Regression R²: 0.6974

Antal dagar per zon:


,Zon,Antal dagar
0,Neutral zon,2626
1,Övre zon,861
2,Nedre zon,617



### Resultat och slutsats – Steg 3

Vi har byggt en logaritmisk regressionsmodell
för Ethereums historiska prisutveckling.

Modellen använder logaritmen av ETH-priset som
målvariabel och antal dagar som förklarande variabel.

**Resultat:**
- Observationer: 4 104.
- R²: 0,6974.
- Residualernas standardavvikelse: 1,203.
- Nedre priszon: 617 dagar.
- Neutral priszon: 2 626 dagar.
- Övre priszon: 861 dagar.

Regressionsmodellen förklarar cirka 69,7 % av
variationen i logaritmerat ETH-pris under den
undersökta perioden.

Det betyder inte att modellen har 69,7 %
prognossäkerhet.

**Slutsats:**

Logaritmisk regression kan beskriva en del av
Ethereums långsiktiga historiska prisutveckling.

Modellen är dock anpassad till hela perioden
2015–2026 och kan därför inte användas direkt
för historisk handelssimulering utan risk för
framtidsinformation.



## Steg 4 – Expanding Window Regression

### Syfte

Vi bygger en historiskt korrekt logaritmisk
regressionsmodell för Ethereum.

Modellen använder endast tidigare kända ETH-priser.

För varje datum:
1. Använd tidigare historiska observationer.
2. Beräkna logaritmisk regression.
3. Beräkna övre och nedre regressionsband.
4. Jämför dagens ETH-pris med banden.
5. Klassificera dagens priszon.

Vi kräver minst 365 tidigare prisobservationer.

Zoner:
- Nedre zon: Pris under nedre regressionsbandet.
- Neutral zon: Pris inom regressionsbanden.
- Övre zon: Pris över övre regressionsbandet.

Modellen används senare för att undersöka
framtida avkastning efter 6 och 12 månader.

En signal baserad på dagens stängningspris
kan tidigast användas för handel nästa dag.


In [6]:

# STEG 4 – Expanding Window Regression för Ethereum

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Utgå från ETH-data från steg 1
df_exp = df_eth.copy()
df_exp = df_exp.sort_values("time").reset_index(drop=True)

df_exp = df_exp[df_exp["ETH_Price"] > 0].copy()
df_exp = df_exp.reset_index(drop=True)

# Dagnummer från första prisobservation
x = (
    (df_exp["time"] - df_exp["time"].iloc[0])
    .dt.total_seconds().to_numpy() / 86400
)

# Logaritmerat ETH-pris
y = np.log(df_exp["ETH_Price"].to_numpy())

# Minst 365 tidigare observationer
min_history = 365

# Kumulativa summor: används för att undvika
# att träna om regressionen tusentals gånger
def previous_sum(values):
    return np.concatenate((
        [0.0],
        np.cumsum(values)[:-1]
    ))

n = np.arange(len(df_exp), dtype=float)

sx = previous_sum(x)
sy = previous_sum(y)
sxx = previous_sum(x * x)
sxy = previous_sum(x * y)
syy = previous_sum(y * y)

# Regressionskoefficienter
denominator = n * sxx - sx ** 2

valid = (
    (n >= min_history)
    & (denominator > 0)
)

slope = np.full(len(df_exp), np.nan)
intercept = np.full(len(df_exp), np.nan)

slope[valid] = (
    (n[valid] * sxy[valid] - sx[valid] * sy[valid])
    / denominator[valid]
)

intercept[valid] = (
    sy[valid] - slope[valid] * sx[valid]
) / n[valid]

# Dagens trendprognos baserad på tidigare dagar
pred_log = intercept + slope * x

# Residualernas historiska standardavvikelse
# SSE från den tidigare anpassade regressionen
sse = syy - intercept * sy - slope * sxy
sse = np.maximum(sse, 0)

residual_std = np.full(len(df_exp), np.nan)
residual_std[valid] = np.sqrt(
    sse[valid] / (n[valid] - 2)
)

# Omvandla logaritmiska värden till USD
df_exp["Regression_Trend"] = np.exp(pred_log)

df_exp["Upper_Band"] = np.exp(
    pred_log + residual_std
)

df_exp["Lower_Band"] = np.exp(
    pred_log - residual_std
)

# Priszoner
df_exp["Price_Zone"] = "INGEN DATA"

df_exp.loc[valid, "Price_Zone"] = "Neutral zon"

df_exp.loc[
    valid
    & (df_exp["ETH_Price"] < df_exp["Lower_Band"]),
    "Price_Zone"
] = "Nedre zon"

df_exp.loc[
    valid
    & (df_exp["ETH_Price"] > df_exp["Upper_Band"]),
    "Price_Zone"
] = "Övre zon"

# Standardiserat avstånd från regression
df_exp["Zone_Score"] = (
    y - pred_log
) / residual_std

# Statistik
print("Antal observationer:", len(df_exp))
print("Minsta träningshistorik:", min_history)

print("\nFörsta giltiga regressionsdatum:")
print(
    df_exp.loc[
        df_exp["Regression_Trend"].notna(),
        "time"
    ].min()
)

print("\nFördelning av priszoner:")
zone_table = (
    df_exp["Price_Zone"]
    .value_counts()
    .rename_axis("Zon")
    .reset_index(name="Antal dagar")
)

zone_table["Andel (%)"] = (
    zone_table["Antal dagar"] / len(df_exp) * 100
).round(2)

display(zone_table)

print("\nSenaste observationerna:")

display(
    df_exp[
        [
            "time",
            "ETH_Price",
            "Regression_Trend",
            "Lower_Band",
            "Upper_Band",
            "Zone_Score",
            "Price_Zone"
        ]
    ].tail(10).round(4)
)


Antal observationer: 4104
Minsta träningshistorik: 365

Första giltiga regressionsdatum:
2016-08-07 00:00:00+00:00

Fördelning av priszoner:


,Zon,Antal dagar,Andel (%)
0,Nedre zon,1991,48.51
1,Neutral zon,1650,40.20
2,INGEN DATA,365,8.89
3,Övre zon,98,2.39



Senaste observationerna:


C:\Users\milen\AppData\Local\Temp\ipykernel_10760\3847591771.py:147: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(4)


,time,ETH_Price,Regression_Trend,Lower_Band,Upper_Band,Zone_Score,Price_Zone
4094,2026-10-23 00:00:00+00:00,2378.2604,10905.6050,3277.5972,36286.4058,-1.2668,Nedre zon
4095,2026-10-24 00:00:00+00:00,2351.9852,10906.2510,3277.5010,36291.7698,-1.2760,Nedre zon
4096,2026-10-25 00:00:00+00:00,2325.7966,10906.7760,3277.3572,36296.8560,-1.2853,Nedre zon
4097,2026-10-26 00:00:00+00:00,2316.5986,10907.1793,3277.1656,36301.6634,-1.2885,Nedre zon
4098,2026-10-27 00:00:00+00:00,2347.5622,10907.5381,3276.9566,36306.3666,-1.2774,Nedre zon
4099,2026-10-28 00:00:00+00:00,2356.9885,10908.0358,3276.8031,36311.3806,-1.2740,Nedre zon
4100,2026-10-29 00:00:00+00:00,2256.0320,10908.5737,3276.6659,36316.4824,-1.3103,Nedre zon
4101,2026-10-30 00:00:00+00:00,2240.2300,10908.6433,3276.3430,36320.5251,-1.3160,Nedre zon
4102,2026-10-31 00:00:00+00:00,2252.0734,10908.6361,3275.9899,36324.3918,-1.3115,Nedre zon
4103,2026-11-01 00:00:00+00:00,2269.9884,10908.6829,3275.6588,36328.3752,-1.3048,Nedre zon


In [7]:

# STEG 4.1 – Visualisera Expanding Window Regression

fig = go.Figure()

# Övre regressionsband
fig.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["Upper_Band"],
    mode="lines",
    name="Övre band",
    line=dict(color="#2ECC71", width=1)
))

# Nedre regressionsband med fyllning
fig.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["Lower_Band"],
    mode="lines",
    name="Nedre band",
    fill="tonexty",
    fillcolor="rgba(46,204,113,0.16)",
    line=dict(color="#2ECC71", width=1)
))

# Regressionslinje
fig.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["Regression_Trend"],
    mode="lines",
    name="Expanding Regression",
    line=dict(color="#F39C12", width=2.5)
))

# ETH-pris
fig.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["ETH_Price"],
    mode="lines",
    name="ETH-pris",
    line=dict(color="#5DADE2", width=1.5)
))

# Markera observationer i nedre zon
lower_zone = df_exp[
    df_exp["Price_Zone"] == "Nedre zon"
]

fig.add_trace(go.Scatter(
    x=lower_zone["time"],
    y=lower_zone["ETH_Price"],
    mode="markers",
    name="Nedre priszon",
    marker=dict(
        color="#FF4B4B",
        size=4,
        opacity=0.65
    )
))

fig.update_layout(
    title="Ethereum – Expanding Window Regression (2015–2026)",
    xaxis_title="Datum",
    yaxis_title="ETH-pris (USD, logaritmisk skala)",
    yaxis_type="log",
    template="plotly_dark",
    hovermode="x unified",
    height=700,
    legend_title="Indikator"
)

fig.update_xaxes(rangeslider_visible=True)

fig.show()



### Resultat och slutsats – Steg 4

Vi har utvecklat en Expanding Window Regression
för att skapa historiskt beräknade priszoner.

Modellen använder minst 365 tidigare prisobservationer
och uppdateras successivt med ny historisk data.

**Resultat:**
- Totalt antal observationer: 4 104.
- Första giltiga regressionsdatum: 2016-08-07.
- Nedre zon: 1 991 dagar (48,51 %).
- Neutral zon: 1 650 dagar (40,20 %).
- Övre zon: 98 dagar (2,39 %).
- Saknade zonvärden: 365 dagar (8,89 %).

Den nedre zonen förekommer betydligt oftare
än den övre zonen.

Vid den sista observationen, 2026-11-01,
var ETH-priset cirka 2 270 USD medan modellens
regressionslinje låg omkring 10 909 USD.

Det visar en stor avvikelse mellan modellens
långsiktiga trend och det faktiska priset.

**Slutsats:**

Expanding Window-metoden gör det möjligt att
beräkna regressionszoner utan att använda priser
från framtida observationer.

Den stora skillnaden mellan regressionslinjen
och senare ETH-priser visar samtidigt att
modellen kan vara känslig för förändrade
marknadsförhållanden.

En nedre priszon innebär därför inte automatiskt
att Ethereum är undervärderat eller köpvärt.



## Steg 5 – Framtida avkastning per ackumulationszon

### Forskningsfråga

Har köp av Ethereum i den nedre regressionszonen
historiskt gett högre avkastning efter 6 och
12 månader än köp i andra zoner?

### Metod

Vi använder zonerna från Expanding Window Regression.

För varje historiskt datum:
1. Identifiera aktuell priszon.
2. Utgå från ett köp nästa dag.
3. Beräkna avkastningen efter 180 dagar.
4. Beräkna avkastningen efter 365 dagar.
5. Jämför avkastningen mellan priszonerna.

Vi analyserar medelvärde, median, andel positiva
utfall och antal observationer.

Detta är en explorativ historisk jämförelse.
Observationerna överlappar varandra och utgör
ännu inte ett oberoende backtest.


In [8]:

# STEG 5 – Framtida avkastning per ETH-priszon

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

df_returns = df_exp.copy()
df_returns = df_returns.sort_values("time").reset_index(drop=True)

# Historiska priser för datumuppslag
price_history = df_returns[
    ["time", "ETH_Price"]
].copy()

# Vi använder endast datum med beräknade zoner
df_returns = df_returns[
    df_returns["Price_Zone"].isin([
        "Nedre zon", "Neutral zon", "Övre zon"
    ])
].copy()

# Nästa dag som simulerad köpdag
entry_lookup = price_history.rename(
    columns={"ETH_Price": "Entry_Price"}
).copy()

entry_lookup["time"] -= pd.Timedelta(days=1)

df_returns = df_returns.merge(
    entry_lookup,
    on="time",
    how="left",
    validate="one_to_one"
)

# Framtida utfall räknas från köpdag (t+1)
horizons = {
    "6M": 180,
    "12M": 365
}

for label, days in horizons.items():

    future_lookup = price_history.rename(
        columns={"ETH_Price": f"Future_Price_{label}"}
    ).copy()

    future_lookup["time"] -= pd.Timedelta(
        days=days + 1
    )

    df_returns = df_returns.merge(
        future_lookup,
        on="time",
        how="left",
        validate="one_to_one"
    )

    df_returns[f"Return_{label}"] = (
        df_returns[f"Future_Price_{label}"]
        / df_returns["Entry_Price"] - 1
    ) * 100

# Sammanfatta per zon och horisont
results = []

for label in horizons:

    return_col = f"Return_{label}"

    for zone in [
        "Nedre zon",
        "Neutral zon",
        "Övre zon"
    ]:

        values = df_returns.loc[
            df_returns["Price_Zone"] == zone,
            return_col
        ].dropna()

        if values.empty:
            continue

        results.append({
            "Horisont": label,
            "Priszon": zone,
            "Observationer": len(values),
            "Medelavkastning (%)": values.mean(),
            "Medianavkastning (%)": values.median(),
            "Andel positiva (%)": (
                (values > 0).mean() * 100
            )
        })

df_zone_results = pd.DataFrame(results)

print("Framtida ETH-avkastning per priszon:")

display(
    df_zone_results.round(2)
)

print("\nSenaste historiska observationer:")

display(
    df_returns[
        [
            "time",
            "ETH_Price",
            "Price_Zone",
            "Entry_Price",
            "Return_6M",
            "Return_12M"
        ]
    ].tail(10).round(2)
)


Framtida ETH-avkastning per priszon:


,Horisont,Priszon,Observationer,Medelavkastning (%),Medianavkastning (%),Andel positiva (%)
0,6M,Nedre zon,1810,171.70,25.54,70.44
1,6M,Neutral zon,1650,95.70,-0.72,49.58
2,6M,Övre zon,98,154.11,146.48,90.82
3,12M,Nedre zon,1635,750.36,61.31,74.68
4,12M,Neutral zon,1640,190.44,-20.49,40.06
5,12M,Övre zon,98,193.37,74.19,80.61



Senaste historiska observationer:


C:\Users\milen\AppData\Local\Temp\ipykernel_10760\4019913508.py:117: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,ETH_Price,Price_Zone,Entry_Price,Return_6M,Return_12M
3729,2026-10-23 00:00:00+00:00,2378.26,Nedre zon,2351.99,NaN,NaN
3730,2026-10-24 00:00:00+00:00,2351.99,Nedre zon,2325.80,NaN,NaN
3731,2026-10-25 00:00:00+00:00,2325.80,Nedre zon,2316.60,NaN,NaN
3732,2026-10-26 00:00:00+00:00,2316.60,Nedre zon,2347.56,NaN,NaN
3733,2026-10-27 00:00:00+00:00,2347.56,Nedre zon,2356.99,NaN,NaN
3734,2026-10-28 00:00:00+00:00,2356.99,Nedre zon,2256.03,NaN,NaN
3735,2026-10-29 00:00:00+00:00,2256.03,Nedre zon,2240.23,NaN,NaN
3736,2026-10-30 00:00:00+00:00,2240.23,Nedre zon,2252.07,NaN,NaN
3737,2026-10-31 00:00:00+00:00,2252.07,Nedre zon,2269.99,NaN,NaN
3738,2026-11-01 00:00:00+00:00,2269.99,Nedre zon,NaN,NaN,NaN


In [9]:

# STEG 5.1 – Interaktiva avkastningsdiagram

import plotly.express as px

zone_order = [
    "Nedre zon",
    "Neutral zon",
    "Övre zon"
]

# Diagram 1 – Medianavkastning
fig_median = px.bar(
    df_zone_results,
    x="Priszon",
    y="Medianavkastning (%)",
    color="Horisont",
    barmode="group",
    text_auto=".1f",
    category_orders={"Priszon": zone_order},
    title="ETH – Medianavkastning efter 6 och 12 månader",
    template="plotly_dark"
)

fig_median.update_layout(
    xaxis_title="Regressionszon",
    yaxis_title="Medianavkastning (%)",
    height=550
)

fig_median.add_hline(
    y=0,
    line_dash="dash"
)

fig_median.show()

# Diagram 2 – Andel positiva utfall
fig_positive = px.bar(
    df_zone_results,
    x="Priszon",
    y="Andel positiva (%)",
    color="Horisont",
    barmode="group",
    text_auto=".1f",
    category_orders={"Priszon": zone_order},
    title="ETH – Sannolikhet för historiskt positiv avkastning",
    template="plotly_dark"
)

fig_positive.update_layout(
    xaxis_title="Regressionszon",
    yaxis_title="Andel positiva historiska utfall (%)",
    yaxis_range=[0, 100],
    height=550
)

fig_positive.show()



### Resultat och slutsats – Steg 5

Vi har undersökt hur Ethereums framtida avkastning
skiljer sig mellan de tre regressionszonerna.

För varje historisk observation identifierades
priszonen och avkastningen beräknades från
nästa dags pris över 180 respektive 365 dagar.

### Resultat – 6 månader

| Priszon | Observationer | Medianavkastning | Positiva utfall |
|---|---:|---:|---:|
| Nedre zon | 1 810 | +25,54 % | 70,44 % |
| Neutral zon | 1 650 | −0,72 % | 49,58 % |
| Övre zon | 98 | +146,48 % | 90,82 % |

### Resultat – 12 månader

| Priszon | Observationer | Medianavkastning | Positiva utfall |
|---|---:|---:|---:|
| Nedre zon | 1 635 | +61,31 % | 74,68 % |
| Neutral zon | 1 640 | −20,49 % | 40,06 % |
| Övre zon | 98 | +74,19 % | 80,61 % |

**Tolkning:**

Den nedre regressionszonen har historiskt
haft positiv medianavkastning efter både
6 och 12 månader.

Den neutrala zonen har däremot haft negativ
medianavkastning för båda horisonterna.

Den övre zonen har haft den högsta
medianavkastningen, vilket inte stödjer
antagandet att den nedre zonen alltid
är den mest gynnsamma köpzonen.

Den övre zonen har dock betydligt färre
observationer än övriga zoner.

**Slutsats:**

Resultaten visar historiska samband mellan
regressionszon och framtida avkastning.

De visar däremot inte att zonerna har
tillförlitlig prognosförmåga.

Framtida avkastningsperioder överlappar,
vilket innebär att observationerna inte
kan betraktas som statistiskt oberoende.



## Steg 6 – Köpzonernas utveckling över tid

Vi undersöker om de historiska sambanden
mellan regressionszon och framtida ETH-avkastning
är stabila över olika kalenderår.

Vi analyserar:

1. Antal observationer i varje zon per år.
2. Medianavkastning efter 6 och 12 månader.
3. Andelen positiva utfall.
4. Skillnader mellan marknadsperioder.

Syftet är att identifiera om resultaten påverkas
av enstaka år med kraftiga prisuppgångar.

Analysen är explorativ. Dagliga observationer
med överlappande framtida avkastning behandlas
inte som oberoende investeringar.


In [10]:

# STEG 6 – Undersök köpzonerna år för år

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

# Använd resultaten från steg 5
df_yearly = df_returns.copy()

df_yearly["År"] = df_yearly["time"].dt.year

zone_order = [
    "Nedre zon",
    "Neutral zon",
    "Övre zon"
]

# TABELL 1 – Antal dagar i varje priszon
zone_counts = pd.crosstab(
    df_yearly["År"],
    df_yearly["Price_Zone"]
)

zone_counts = zone_counts.reindex(
    columns=zone_order,
    fill_value=0
)

print("Antal observationer i varje zon per år:")
display(zone_counts)

# DIAGRAM 1 – Priszonernas fördelning
zone_chart = zone_counts.reset_index().melt(
    id_vars="År",
    var_name="Priszon",
    value_name="Antal dagar"
)

fig1 = px.bar(
    zone_chart,
    x="År",
    y="Antal dagar",
    color="Priszon",
    category_orders={"Priszon": zone_order},
    barmode="stack",
    title="Ethereum – Priszonernas fördelning per år",
    template="plotly_dark",
    color_discrete_map={
        "Nedre zon": "#2ECC71",
        "Neutral zon": "#F39C12",
        "Övre zon": "#FF4B4B"
    }
)

fig1.update_layout(height=550)
fig1.show()

# TABELL 2 – Framtida avkastning per år och zon
yearly_results = []

for horizon in ["6M", "12M"]:

    return_col = f"Return_{horizon}"

    for (year, zone), group in df_yearly.groupby(
        ["År", "Price_Zone"]
    ):

        values = group[return_col].dropna()

        if values.empty:
            continue

        yearly_results.append({
            "År": year,
            "Priszon": zone,
            "Horisont": horizon,
            "Observationer": len(values),
            "Medianavkastning (%)": values.median(),
            "Medelavkastning (%)": values.mean(),
            "Andel positiva (%)": (
                values.gt(0).mean() * 100
            )
        })

df_yearly_results = pd.DataFrame(yearly_results)

print("\nAvkastning per år och priszon:")
display(df_yearly_results.round(2))

# DIAGRAM 2 – Heatmap över medianavkastning
for horizon in ["6M", "12M"]:

    subset = df_yearly_results[
        df_yearly_results["Horisont"] == horizon
    ]

    heatmap_data = subset.pivot(
        index="År",
        columns="Priszon",
        values="Medianavkastning (%)"
    ).reindex(columns=zone_order)

    fig2 = px.imshow(
        heatmap_data,
        text_auto=".1f",
        aspect="auto",
        color_continuous_scale="RdYlGn",
        color_continuous_midpoint=0,
        title=(
            "Ethereum – Medianavkastning per år "
            f"och köpzon ({horizon})"
        ),
        labels={
            "color": "Avkastning (%)",
            "x": "Priszon",
            "y": "År"
        }
    )

    fig2.update_layout(
        template="plotly_dark",
        height=550
    )

    fig2.show()


Antal observationer i varje zon per år:


Price_Zone,Nedre zon,Neutral zon,Övre zon
År,,,
2016,144,3,0
2017,44,232,89
2018,251,105,9
2019,365,0,0
2020,185,181,0
2021,0,365,0
2022,161,204,0
2023,240,125,0
2024,83,283,0



Avkastning per år och priszon:


,År,Priszon,Horisont,Observationer,Medianavkastning (%),Medelavkastning (%),Andel positiva (%)
0,2016,Nedre zon,6M,144,310.06,1151.58,97.22
1,2016,Neutral zon,6M,3,203.75,139.59,66.67
2,2017,Nedre zon,6M,44,2047.58,2108.36,100.00
3,2017,Neutral zon,6M,232,153.10,424.04,90.95
4,2017,Övre zon,6M,89,158.26,176.09,100.00
5,2018,Nedre zon,6M,251,-43.79,-16.23,19.52
6,2018,Neutral zon,6M,105,-66.55,-63.79,0.00
7,2018,Övre zon,6M,9,-62.27,-63.25,0.00
8,2019,Nedre zon,6M,365,12.12,16.67,64.38
9,2020,Nedre zon,6M,185,116.54,137.80,100.00



### Resultat och slutsats – Steg 6

Vi har undersökt hur regressionszonernas
förekomst och framtida avkastning varierar
mellan olika kalenderår.

**Resultat:**

- 2019: Samtliga 365 dagar klassificerades
  som nedre zon.
- 2021: Samtliga 365 dagar klassificerades
  som neutral zon.
- 2026: Samtliga 305 observerade dagar
  klassificerades som nedre zon.
- Övre zonen förekom nästan enbart under
  2017 och 2018.

Årsvisa avkastningsresultat visar stora skillnader.

Exempelvis var medianavkastningen i nedre zonen:

| År | 6 månader | 12 månader |
|---|---:|---:|
| 2017 | +2 047,6 % | +9 777,9 % |
| 2018 | −43,8 % | −29,3 % |
| 2022 | +35,2 % | +52,9 % |
| 2024 | −23,4 % | +67,4 % |

**Tolkning:**

Regressionszonernas förekomst varierar kraftigt
mellan olika marknadsperioder.

De mycket höga avkastningarna under 2017
påverkar de historiska sammanställningarna.

Samtidigt visar 2018 att ett köp i nedre
zonen även kan följas av betydande förluster.

**Slutsats:**

Den nedre zonen har inte konsekvent varit
en gynnsam köpzon under alla marknadsår.

Modellens resultat påverkas av marknadscykler
och bör därför utvärderas över flera
tidsperioder.



## Steg 7 – Tidsbaserad validering av köpzoner

Vi utvärderar om regressionszonerna har ett
stabilt samband med framtida ETH-avkastning.

Metod:
- Expanding Window
- Årliga valideringsperioder
- 6 och 12 månaders prognoshorisont
- Endast historiskt tillgängliga regressionsband

Vi jämför köp i nedre zonen med en referens:
köp på alla giltiga datum under samma år.

Tidigare utfall får bara ingå i träningshistoriken
om de var kända före valideringsperiodens start.

Vi ändrar inte zonernas definition baserat
på valideringsresultaten.

Resultaten är explorativa eftersom avkastningarna
överlappar och vi redan har granskat historiska data.


In [11]:

# STEG 7 – Årlig Expanding Window-utvärdering

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

df_cv = df_returns.copy()

df_cv["time"] = pd.to_datetime(
    df_cv["time"], utc=True
)

df_cv["Entry_Date"] = (
    df_cv["time"] + pd.Timedelta(days=1)
)

results_cv = []

# Validering år för år
validation_years = range(2019, 2025)

for horizon, days in [("6M", 180), ("12M", 365)]:

    return_col = f"Return_{horizon}"

    df_cv["Exit_Date"] = (
        df_cv["Entry_Date"]
        + pd.Timedelta(days=days)
    )

    for year in validation_years:

        start = pd.Timestamp(
            f"{year}-01-01", tz="UTC"
        )

        end = pd.Timestamp(
            f"{year + 1}-01-01", tz="UTC"
        )

        # Historik med kända utfall före validering
        train = df_cv[
            (df_cv["Exit_Date"] < start)
            & df_cv[return_col].notna()
        ]

        # Valideringsperiod
        val = df_cv[
            (df_cv["time"] >= start)
            & (df_cv["time"] < end)
            & df_cv[return_col].notna()
        ].copy()

        if val.empty:
            continue

        # Referens: köp på samtliga giltiga datum
        baseline_median = val[return_col].median()

        for zone in [
            "Nedre zon",
            "Neutral zon",
            "Övre zon"
        ]:

            zone_data = val[
                val["Price_Zone"] == zone
            ]

            if zone_data.empty:
                continue

            values = zone_data[return_col]

            results_cv.append({
                "År": year,
                "Horisont": horizon,
                "Priszon": zone,
                "Träningshistorik": len(train),
                "Observationer": len(values),
                "Medianavkastning": values.median(),
                "Baseline_Median": baseline_median,
                "Skillnad_mot_Baseline": (
                    values.median() - baseline_median
                ),
                "Andel_Positiva": (
                    values.gt(0).mean() * 100
                )
            })

df_cv_results = pd.DataFrame(results_cv)

print("Tidsbaserad utvärdering per år:")

display(
    df_cv_results.round(2)
)


Tidsbaserad utvärdering per år:


,År,Horisont,Priszon,Träningshistorik,Observationer,Medianavkastning,Baseline_Median,Skillnad_mot_Baseline,Andel_Positiva
0,2019,6M,Nedre zon,696,365,12.12,12.12,0.00,64.38
1,2020,6M,Nedre zon,1061,185,116.54,205.21,-88.67,100.00
2,2020,6M,Neutral zon,1061,181,361.41,205.21,156.20,100.00
3,2021,6M,Neutral zon,1427,365,42.50,42.50,0.00,61.64
4,2022,6M,Nedre zon,1792,161,35.24,-4.28,39.52,93.79
5,2022,6M,Neutral zon,1792,204,-39.16,-4.28,-34.89,6.37
6,2023,6M,Nedre zon,2157,240,31.51,26.23,5.27,93.75
7,2023,6M,Neutral zon,2157,125,18.62,26.23,-7.62,63.20
8,2024,6M,Nedre zon,2522,83,-23.42,-14.81,-8.61,15.66
9,2024,6M,Neutral zon,2522,283,-11.82,-14.81,2.99,32.16


In [12]:

# STEG 7.1 – Visualisera valideringsresultaten

# Diagram 1 – Skillnad mot referens
fig_cv = px.bar(
    df_cv_results,
    x="År",
    y="Skillnad_mot_Baseline",
    color="Priszon",
    barmode="group",
    facet_row="Horisont",
    title="ETH – Priszonernas medianavkastning jämfört med referens",
    labels={
        "Skillnad_mot_Baseline": "Skillnad (procentenheter)"
    },
    template="plotly_dark"
)

fig_cv.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

fig_cv.update_layout(
    height=750
)

fig_cv.show()

# Diagram 2 – Positiva utfall per år
fig_positive_cv = px.line(
    df_cv_results,
    x="År",
    y="Andel_Positiva",
    color="Priszon",
    facet_row="Horisont",
    markers=True,
    title="ETH – Positiva framtida utfall per zon och år",
    labels={
        "Andel_Positiva": "Andel positiva (%)"
    },
    template="plotly_dark"
)

fig_positive_cv.update_layout(
    height=750
)

fig_positive_cv.update_yaxes(
    range=[0, 100]
)

fig_positive_cv.show()



### Resultat och slutsats – Steg 7

Vi har genomfört en årsvis historisk utvärdering
av regressionszonernas framtida avkastning.

För varje år jämfördes medianavkastningen
för respektive zon med medianavkastningen
från köp på samtliga giltiga datum samma år.

### Nedre zon jämfört med referens

| År | Skillnad 6M | Skillnad 12M |
|---|---:|---:|
| 2019 | 0,00 pp | 0,00 pp |
| 2020 | −88,67 pp | +96,53 pp |
| 2022 | +39,52 pp | +48,37 pp |
| 2023 | +5,27 pp | −2,59 pp |
| 2024 | −8,61 pp | +74,90 pp |

pp = procentenheter.

År 2021 saknades observationer i nedre zonen.

**Tolkning:**

Resultaten är blandade för 6-månadershorisonten.

För 12 månader har nedre zonen överträffat
referensen under flera år, men inte samtliga.

År där endast en zon förekommer ger begränsad
information om modellens förmåga att skilja
mellan gynnsamma och ogynnsamma köpzoner.

**Metodbegränsning:**

Den genomförda årsjämförelsen är inte en
fullständig Expanding Window Cross-Validation
för modellval.

Vi använder historiskt beräknade zoner,
men har ännu inte genomfört en separat
modellvalsprocess med successiva tränings-
och valideringsfönster.

**Slutsats:**

Regressionszonerna visar viss historisk
förmåga att skilja mellan olika avkastningsutfall.

Resultaten är dock inte tillräckligt konsekventa
för att fastställa att modellen identifierar
gynnsamma köpzoner på nya data.



## Steg 8 – Backtesting av ETH:s ackumulationszoner

### Syfte

Vi undersöker om en regelbaserad strategi
baserad på nedre regressionszonen historiskt
hade presterat bättre än Buy & Hold.

Startkapital: 10 000 USD.

Strategi:
- Nedre zon: Äg Ethereum.
- Neutral eller övre zon: Behåll kontanter.
- Positionen ändras med en dags exekveringsfördröjning.

Handelskostnad: 0,1 % per positionsförändring.
Ingen hävstång och ingen ränta på kontanter.

Vi jämför:
1. Total avkastning
2. Slutkapital
3. Maximal drawdown
4. Antal köp och försäljningar
5. Kapitalutveckling över tid

Detta är ett förenklat historiskt backtest.
Strategin är ännu inte oberoende validerad.


In [13]:

# STEG 8 – Backtesting av ETH:s ackumulationszoner

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

START_CAPITAL = 10000
FEE = 0.001  # 0,1 % per köp eller försäljning

# Vi jämför samma historiska period
bt = df_exp[
    df_exp["time"] >= pd.Timestamp(
        "2019-01-01", tz="UTC"
    )
].copy()

bt = bt.sort_values("time").reset_index(drop=True)

# Daglig ETH-avkastning
bt["ETH_Return"] = (
    bt["ETH_Price"].pct_change()
)

# Enbart nedre zon ger innehav
bt["Buy_Condition"] = (
    bt["Price_Zone"] == "Nedre zon"
).astype(int)

# Konservativ tidsförskjutning:
# Signal vid stängning dag t kan verkställas
# vid nästa dags stängning, och positionen
# får avkastning först efter det.
bt["Position"] = (
    bt["Buy_Condition"].shift(2).fillna(0)
).astype(int)

# Strategins dagliga avkastning
bt["Gross_Return"] = (
    bt["Position"] * bt["ETH_Return"].fillna(0)
)

# Positionsförändringar
bt["Trade"] = (
    bt["Position"].diff().fillna(bt["Position"])
)

# Handelskostnad vid ändrad position
bt["Trading_Cost"] = bt["Trade"].abs() * FEE

bt["Strategy_Return"] = (
    bt["Gross_Return"] - bt["Trading_Cost"]
)

# Buy & Hold: samma startperiod
bt["BuyHold_Return"] = (
    bt["ETH_Return"].fillna(0)
)

# Kapitalutveckling
bt["Strategy_Value"] = (
    START_CAPITAL
    * (1 + bt["Strategy_Return"]).cumprod()
)

bt["BuyHold_Value"] = (
    START_CAPITAL
    * (1 + bt["BuyHold_Return"]).cumprod()
)

# Drawdown
for prefix, column in [
    ("Strategy", "Strategy_Value"),
    ("BuyHold", "BuyHold_Value")
]:
    peak = bt[column].cummax().clip(
        lower=START_CAPITAL
    )

    bt[f"{prefix}_Drawdown"] = (
        bt[column] / peak - 1
    ) * 100

# Resultattabell
summary_bt = pd.DataFrame([
    {
        "Strategi": "Ackumulationszon",
        "Slutkapital USD": bt["Strategy_Value"].iloc[-1],
        "Total avkastning (%)": (
            bt["Strategy_Value"].iloc[-1]
            / START_CAPITAL - 1
        ) * 100,
        "Max Drawdown (%)": bt[
            "Strategy_Drawdown"
        ].min()
    },
    {
        "Strategi": "Buy & Hold",
        "Slutkapital USD": bt["BuyHold_Value"].iloc[-1],
        "Total avkastning (%)": (
            bt["BuyHold_Value"].iloc[-1]
            / START_CAPITAL - 1
        ) * 100,
        "Max Drawdown (%)": bt[
            "BuyHold_Drawdown"
        ].min()
    }
])

print("Backtesting – Ethereum")
print("Period:", bt["time"].min(), "till", bt["time"].max())
print("Antal köp:", (bt["Trade"] == 1).sum())
print("Antal försäljningar:", (bt["Trade"] == -1).sum())
print(
    "Andel dagar investerad:",
    round(bt["Position"].mean() * 100, 2), "%"
)

display(summary_bt.round(2))


Backtesting – Ethereum
Period: 2019-01-01 00:00:00+00:00 till 2026-11-01 00:00:00+00:00
Antal köp: 45
Antal försäljningar: 44
Andel dagar investerad: 54.16 %


,Strategi,Slutkapital USD,Total avkastning (%),Max Drawdown (%)
0,Ackumulationszon,37457.03,274.57,-63.62
1,Buy & Hold,163127.03,1531.27,-79.36


In [14]:

# STEG 8.1 – Diagram för backtesting

# DIAGRAM 1 – Kapitalutveckling
fig_capital = go.Figure()

fig_capital.add_trace(go.Scatter(
    x=bt["time"],
    y=bt["Strategy_Value"],
    name="Ackumulationsstrategi",
    mode="lines",
    line=dict(color="#2ECC71", width=2)
))

fig_capital.add_trace(go.Scatter(
    x=bt["time"],
    y=bt["BuyHold_Value"],
    name="Buy & Hold",
    mode="lines",
    line=dict(color="#5DADE2", width=2)
))

fig_capital.add_hline(
    y=START_CAPITAL,
    line_dash="dash",
    line_color="gray"
)

fig_capital.update_layout(
    title="Ethereum – Kapitalutveckling: Strategi vs Buy & Hold",
    xaxis_title="Datum",
    yaxis_title="Portföljvärde (USD)",
    template="plotly_dark",
    height=600,
    hovermode="x unified"
)

fig_capital.update_xaxes(rangeslider_visible=True)
fig_capital.show()

# DIAGRAM 2 – Maximal drawdown över tid
fig_risk = go.Figure()

fig_risk.add_trace(go.Scatter(
    x=bt["time"],
    y=bt["Strategy_Drawdown"],
    name="Ackumulationsstrategi",
    mode="lines",
    line=dict(color="#2ECC71")
))

fig_risk.add_trace(go.Scatter(
    x=bt["time"],
    y=bt["BuyHold_Drawdown"],
    name="Buy & Hold",
    mode="lines",
    line=dict(color="#FF4B4B")
))

fig_risk.update_layout(
    title="Ethereum – Drawdown och historisk risk",
    xaxis_title="Datum",
    yaxis_title="Drawdown (%)",
    template="plotly_dark",
    height=500,
    hovermode="x unified"
)

fig_risk.show()



### Resultat och slutsats – Steg 8

Vi har genomfört en historisk handelssimulering
för Ethereum från 2019-01-01 till 2026-11-01.

Startkapitalet var 10 000 USD.

Strategin innebar att kapitalet placerades
i Ethereum när modellen identifierade nedre
regressionszonen, medan kapitalet annars
behölls kontant.

En dags exekveringsfördröjning användes
med ytterligare förskjutning av avkastningen
för att undvika handel på ej tillgängliga priser.

En antagen transaktionskostnad på 0,1 %
användes vid positionsförändringar.

### Resultat

| Mått | Ackumulationsstrategi | Buy & Hold |
|---|---:|---:|
| Startkapital | 10 000 USD | 10 000 USD |
| Slutkapital | 37 457 USD | 163 127 USD |
| Total avkastning | +274,57 % | +1 531,27 % |
| Maximal drawdown | −63,62 % | −79,36 % |

### Strategins aktivitet

- Antal köp: 45.
- Antal försäljningar: 44.
- Andel dagar investerad: 54,16 %.

**Tolkning:**

Buy & Hold gav betydligt högre total avkastning
än den regelbaserade ackumulationsstrategin.

Ackumulationsstrategin uppvisade däremot
en mindre maximal drawdown.

Det innebär att den regelbaserade strategin
minskade den största historiska kapitalnedgången,
men samtidigt missade stora delar av
Ethereums långsiktiga prisuppgång.

**Slutsats:**

Backtestingen ger inget stöd för att den
testade ackumulationsstrategin överträffar
Buy & Hold i total avkastning.

Resultatet visar att en lägre historisk risk
kan medföra betydande alternativkostnader
i form av missad avkastning.

Backtestingen bygger på förenklade antaganden
och utgör inte ett oberoende test av modellen.
Den inkluderar inte slippage eller skatt.



## Steg 9 – Strategins stabilitet över olika marknadsperioder

### Syfte

Vi undersöker om vår ackumulationsstrategi
presterar konsekvent under olika marknadsförhållanden.

Vi delar upp historiken i tre perioder:

- 2019–2022
- 2023–2024
- 2025–2026

### Metod

Vi jämför vår regressionsbaserade ackumulationsstrategi
med Buy & Hold under samma tidsperioder.

För varje period beräknas:

1. Total avkastning (%)
2. Maximal drawdown (%)
3. Antal köp och försäljningar
4. Andel dagar investerad (%)

Syftet är att bedöma strategins historiska
stabilitet och undersöka under vilka
marknadsförhållanden den fungerar bättre
eller sämre än Buy & Hold.

### Hypotes

En strategi baserad på nedre regressionszonen
kan minska exponeringen under vissa
marknadsperioder, men riskerar samtidigt
att missa betydande prisuppgångar.

Detta är en explorativ historisk analys
och inte en oberoende out-of-sample-utvärdering.


In [15]:

# STEG 9 – Backtesting per marknadsperiod

import pandas as pd
import numpy as np
from IPython.display import display

periods = {
    "2019–2022": ("2019-01-01", "2023-01-01"),
    "2023–2024": ("2023-01-01", "2025-01-01"),
    "2025–2026": ("2025-01-01", "2026-11-02")
}

period_results = []

for period_name, (start, end) in periods.items():

    subset = bt[
        (bt["time"] >= pd.Timestamp(start, tz="UTC"))
        & (bt["time"] < pd.Timestamp(end, tz="UTC"))
    ].copy()

    if subset.empty:
        continue

    for strategy, return_col in [
        ("Ackumulationsstrategi", "Strategy_Return"),
        ("Buy & Hold", "BuyHold_Return")
    ]:

        returns = subset[return_col].fillna(0)

        # Kapitalutveckling inom perioden
        equity = (1 + returns).cumprod()

        # Maximal drawdown
        peak = equity.cummax().clip(lower=1.0)
        drawdown = (equity / peak - 1) * 100

        total_return = (equity.iloc[-1] - 1) * 100

        if strategy == "Ackumulationsstrategi":

            invested_pct = (
                subset["Position"].mean() * 100
            )

            buys = int((subset["Trade"] == 1).sum())
            sells = int((subset["Trade"] == -1).sum())

        else:
            invested_pct = 100.0
            buys = 1
            sells = 0

        period_results.append({
            "Period": period_name,
            "Strategi": strategy,
            "Avkastning (%)": total_return,
            "Max Drawdown (%)": drawdown.min(),
            "Investerad tid (%)": invested_pct,
            "Antal köp": buys,
            "Antal försäljningar": sells
        })

df_period_results = pd.DataFrame(period_results)

print("RESULTAT – BACKTESTING PER PERIOD")

display(
    df_period_results.round(2)
)


RESULTAT – BACKTESTING PER PERIOD


,Period,Strategi,Avkastning (%),Max Drawdown (%),Investerad tid (%),Antal köp,Antal försäljningar
0,2019–2022,Ackumulationsstrategi,140.39,-63.62,48.53,19,18
1,2019–2022,Buy & Hold,758.99,-79.36,100.00,1,0
2,2023–2024,Ackumulationsstrategi,107.04,-22.51,44.46,22,23
3,2023–2024,Buy & Hold,178.79,-45.48,100.00,1,0
4,2025–2026,Ackumulationsstrategi,-24.74,-53.39,77.01,4,3
5,2025–2026,Buy & Hold,-31.88,-67.62,100.00,1,0


In [16]:

# STEG 9.1 – Visualisering av strategins stabilitet

import plotly.express as px

strategy_colors = {
    "Ackumulationsstrategi": "#2ECC71",
    "Buy & Hold": "#5DADE2"
}

# DIAGRAM 1 – Avkastning per period
fig_return = px.bar(
    df_period_results,
    x="Period",
    y="Avkastning (%)",
    color="Strategi",
    barmode="group",
    text_auto=".1f",
    color_discrete_map=strategy_colors,
    title="Ethereum – Avkastning per marknadsperiod",
    template="plotly_dark"
)

fig_return.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

fig_return.update_layout(
    height=550,
    xaxis_title="Marknadsperiod",
    yaxis_title="Total avkastning (%)"
)

fig_return.show()

# DIAGRAM 2 – Max Drawdown
fig_drawdown = px.bar(
    df_period_results,
    x="Period",
    y="Max Drawdown (%)",
    color="Strategi",
    barmode="group",
    text_auto=".1f",
    color_discrete_map=strategy_colors,
    title="Ethereum – Maximal drawdown per period",
    template="plotly_dark"
)

fig_drawdown.update_layout(
    height=550,
    xaxis_title="Marknadsperiod",
    yaxis_title="Maximal drawdown (%)"
)

fig_drawdown.show()

# DIAGRAM 3 – Andel investerad tid
strategy_only = df_period_results[
    df_period_results["Strategi"]
    == "Ackumulationsstrategi"
]

fig_invested = px.bar(
    strategy_only,
    x="Period",
    y="Investerad tid (%)",
    text_auto=".1f",
    title="Ethereum – Andel dagar investerad per period",
    template="plotly_dark",
    color_discrete_sequence=["#2ECC71"]
)

fig_invested.update_layout(
    height=500,
    yaxis_range=[0, 100],
    xaxis_title="Marknadsperiod",
    yaxis_title="Investerad tid (%)"
)

fig_invested.show()



### Resultat och slutsats – Steg 9

#### Syfte

Vi undersökte hur vår regressionsbaserade
ackumulationsstrategi presterade under olika
marknadsperioder.

Strategin jämfördes med Buy & Hold under
tre perioder mellan 2019 och 2026.

#### Resultat – Avkastning

| Period | Ackumulationsstrategi | Buy & Hold |
|---|---:|---:|
| 2019–2022 | +140,39 % | +758,99 % |
| 2023–2024 | +107,04 % | +178,79 % |
| 2025–2026 | −24,74 % | −31,88 % |

Buy & Hold genererade högre avkastning under
de två första perioderna.

Under 2025–2026 presterade ackumulationsstrategin
bättre genom att begränsa förlusten med
7,14 procentenheter jämfört med Buy & Hold.

#### Resultat – Maximal drawdown

| Period | Ackumulationsstrategi | Buy & Hold |
|---|---:|---:|
| 2019–2022 | −63,62 % | −79,36 % |
| 2023–2024 | −22,51 % | −45,48 % |
| 2025–2026 | −53,39 % | −67,62 % |

Ackumulationsstrategin uppvisade lägre maximal
drawdown under samtliga tre perioder.

Det innebär att strategin historiskt begränsade
den största nedgången från tidigare kapitaltopp,
jämfört med Buy & Hold.

Strategin innebar dock fortfarande betydande risk.

#### Resultat – Investerad tid

| Period | Investerad tid |
|---|---:|
| 2019–2022 | 48,53 % |
| 2023–2024 | 44,46 % |
| 2025–2026 | 77,01 % |

Under 2019–2024 var strategin investerad
mindre än hälften av tiden.

Under 2025–2026 ökade exponeringen till 77,01 %,
vilket också innebar att strategin påverkades
mer av Ethereums negativa prisutveckling.

#### Analys

Resultaten visar ett samband mellan
marknadsexponering, avkastning och risk.

Under perioder med kraftiga prisuppgångar
underpresterade ackumulationsstrategin
jämfört med Buy & Hold.

Under den senaste perioden, 2025–2026,
minskade strategin förlusten jämfört med
Buy & Hold.

En möjlig förklaring är att strategin
höll en del av kapitalet kontant när den
nedre regressionszonen inte var aktiv.

Resultaten visar dock inte om riskminskningen
berodde på modellens förmåga att identifiera
gynnsamma priszoner eller enbart på
minskad marknadsexponering.

#### Metodologiska begränsningar

- Resultaten kommer från historisk simulering.
- Strategins regler har analyserats med historiska data.
- Backtestingen är inte ett oberoende test.
- Avkastningen bygger på förenklade handelskostnader.
- Slippage och skatt ingår inte.
- Lägre drawdown innebär inte automatiskt
  bättre riskjusterad avkastning.
- Strategin har inte validerats för handel i realtid.

#### Slutsats

Ackumulationsstrategin gav lägre maximal
drawdown än Buy & Hold under alla tre perioderna.

Däremot gav Buy & Hold betydligt högre avkastning
under de två första perioderna.

Under 2025–2026 överträffade ackumulationsstrategin
Buy & Hold genom att begränsa kapitalförlusten.

Resultaten tyder på att regressionsbaserade
priszoner kan användas för att styra
marknadsexponering och historiskt minska risk
under vissa marknadsförhållanden.

Vi kan däremot inte fastställa att modellen
identifierar framtida gynnsamma köpzoner
med tillräcklig tillförlitlighet.

Ytterligare tidsbaserad validering krävs
innan slutsatser om prognosförmåga kan dras.



## Steg 10 – Expanding Window Cross-Validation

### Syfte

Vi undersöker om historiska regressionszoner
kan identifiera perioder med gynnsam framtida
avkastning för Ethereum.

### Metod

Vi använder Expanding Window Validation.

För varje valideringsperiod:

1. Använd tidigare historiska observationer.
2. Säkerställ att historiska framtidsutfall
   är kända före valideringsperiodens start.
3. Identifiera regressionszoner med den
   tidigare beräknade modellen.
4. Mät framtida avkastning efter 180 och
   365 dagar.
5. Jämför nedre zonen med samtliga giltiga
   köpdatum under samma period.

Vi använder åren 2021–2024 för jämförelsen.

Perioden 2025–2026 redovisas separat och
används inte för parameterjustering.

Vi använder samma fasta zondefinition
genom hela experimentet.

### Begränsning

Denna analys använder en redan fastställd
regelbaserad zonmodell.

Den är inte en fullständig ML-Cross-Validation
med hyperparameteroptimering.

Historiska resultat från tidigare steg har
redan granskats, vilket begränsar möjligheten
att kalla senare perioder helt oberoende.


In [17]:

# STEG 10 – Expanding Window Cross-Validation
# ETH – 6 och 12 månader

import pandas as pd
import numpy as np
from IPython.display import display

cv_data = df_returns.copy()
cv_data["time"] = pd.to_datetime(
    cv_data["time"], utc=True
)

cv_data = cv_data.sort_values("time").reset_index(drop=True)

# Samma definitioner som i steg 5:
# Köp dagen efter signalen
horizons = {"6M": 180, "12M": 365}
validation_years = [2021, 2022, 2023, 2024]

# Undvik att använda 2025 års priser för
# valideringens framtida utfall.
validation_cutoff = pd.Timestamp(
    "2025-01-01", tz="UTC"
)

all_predictions = []
fold_results = []

for horizon, days in horizons.items():

    return_col = f"Return_{horizon}"

    data = cv_data[
        ["time", "Price_Zone", return_col]
    ].dropna().copy()

    data["Entry_Date"] = (
        data["time"] + pd.Timedelta(days=1)
    )

    data["Exit_Date"] = (
        data["Entry_Date"]
        + pd.Timedelta(days=days)
    )

    for year in validation_years:

        start = pd.Timestamp(
            f"{year}-01-01", tz="UTC"
        )

        end = pd.Timestamp(
            f"{year+1}-01-01", tz="UTC"
        )

        # Endast utfall som var kända
        # före valideringsperiodens start
        train = data[
            data["Exit_Date"] < start
        ].copy()

        # Observationer från valideringsåret
        validation = data[
            (data["time"] >= start)
            & (data["time"] < end)
            & (data["Exit_Date"] < validation_cutoff)
        ].copy()

        if len(train) < 100 or validation.empty:
            continue

        # Referensprognos: median för all träningsdata
        baseline_prediction = train[
            return_col
        ].median()

        # Historisk medianavkastning per priszon
        zone_medians = train.groupby(
            "Price_Zone"
        )[return_col].median()

        # Förutsäg framtida avkastning baserat på zon
        validation["Zone_Prediction"] = (
            validation["Price_Zone"]
            .map(zone_medians)
            .fillna(baseline_prediction)
        )

        validation["Baseline_Prediction"] = (
            baseline_prediction
        )

        validation["Actual_Return"] = (
            validation[return_col]
        )

        # Prognosfel
        validation["Zone_Error"] = (
            validation["Actual_Return"]
            - validation["Zone_Prediction"]
        ).abs()

        validation["Baseline_Error"] = (
            validation["Actual_Return"]
            - validation["Baseline_Prediction"]
        ).abs()

        validation["Horisont"] = horizon
        validation["Valideringsår"] = year

        all_predictions.append(validation)

        fold_results.append({
            "Horisont": horizon,
            "År": year,
            "Träning": len(train),
            "Validering": len(validation),
            "MAE_Zonmodell": validation[
                "Zone_Error"
            ].mean(),
            "MAE_Baseline": validation[
                "Baseline_Error"
            ].mean(),
            "Förbättring_MAE": (
                validation["Baseline_Error"].mean()
                - validation["Zone_Error"].mean()
            )
        })

df_cv10 = pd.DataFrame(fold_results)

df_cv10_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

print("EXPANDING WINDOW – RESULTAT")
display(df_cv10.round(2))

print("\nSAMMANFATTNING PER HORISONT")

summary_cv10 = (
    df_cv10_predictions.groupby("Horisont")
    .agg(
        Observationer=("Actual_Return", "size"),
        MAE_Zonmodell=("Zone_Error", "mean"),
        MAE_Baseline=("Baseline_Error", "mean")
    )
    .reset_index()
)

summary_cv10["Förbättring_MAE"] = (
    summary_cv10["MAE_Baseline"]
    - summary_cv10["MAE_Zonmodell"]
)

display(summary_cv10.round(2))


EXPANDING WINDOW – RESULTAT


,Horisont,År,Träning,Validering,MAE_Zonmodell,MAE_Baseline,Förbättring_MAE
0,6M,2021,1427,365,61.60,50.47,-11.12
1,6M,2022,1792,365,88.85,81.94,-6.92
2,6M,2023,2157,365,35.65,36.89,1.24
3,6M,2024,2522,185,33.58,40.00,6.43
4,12M,2021,1242,365,51.86,70.94,19.08
5,12M,2022,1607,365,142.92,98.51,-44.40
6,12M,2023,1972,365,58.53,19.07,-39.46



SAMMANFATTNING PER HORISONT


,Horisont,Observationer,MAE_Zonmodell,MAE_Baseline,Förbättring_MAE
0,12M,1095,84.44,62.84,-21.60
1,6M,1280,57.92,54.06,-3.86


In [18]:

# STEG 10.1 – Visualisera Cross-Validation

import plotly.express as px
import plotly.graph_objects as go

# DIAGRAM 1 – MAE per år
plot_cv = df_cv10.melt(
    id_vars=["År", "Horisont"],
    value_vars=["MAE_Zonmodell", "MAE_Baseline"],
    var_name="Modell",
    value_name="MAE"
)

plot_cv["Modell"] = plot_cv["Modell"].replace({
    "MAE_Zonmodell": "Regressionszoner",
    "MAE_Baseline": "Referensmodell"
})

fig1 = px.bar(
    plot_cv,
    x="År",
    y="MAE",
    color="Modell",
    barmode="group",
    facet_row="Horisont",
    title="Ethereum – Prognosfel per valideringsår",
    template="plotly_dark",
    labels={"MAE": "MAE (procentenheter)"}
)

fig1.update_xaxes(type="category")
fig1.update_layout(height=750)
fig1.show()

# DIAGRAM 2 – Sammanfattning per horisont
summary_plot = summary_cv10.melt(
    id_vars="Horisont",
    value_vars=["MAE_Zonmodell", "MAE_Baseline"],
    var_name="Modell",
    value_name="MAE"
)

fig2 = px.bar(
    summary_plot,
    x="Horisont",
    y="MAE",
    color="Modell",
    barmode="group",
    text_auto=".1f",
    title="Ethereum – Samlat prognosfel: 6M vs 12M",
    template="plotly_dark"
)

fig2.update_layout(
    height=550,
    yaxis_title="MAE (procentenheter)"
)

fig2.show()

# DIAGRAM 3 – Faktiskt vs förutsagt
fig3 = px.scatter(
    df_cv10_predictions,
    x="Zone_Prediction",
    y="Actual_Return",
    color="Horisont",
    opacity=0.4,
    title="Ethereum – Faktisk kontra förutsagd avkastning",
    labels={
        "Zone_Prediction": "Förutsagd avkastning (%)",
        "Actual_Return": "Faktisk avkastning (%)"
    },
    template="plotly_dark"
)

all_values = pd.concat([
    df_cv10_predictions["Zone_Prediction"],
    df_cv10_predictions["Actual_Return"]
])

low = all_values.min()
high = all_values.max()

fig3.add_trace(go.Scatter(
    x=[low, high],
    y=[low, high],
    mode="lines",
    name="Perfekt prognos",
    line=dict(dash="dash", color="white")
))

fig3.update_layout(height=650)
fig3.show()



### Resultat och slutsats – Steg 10

#### Syfte

Vi undersökte om regressionsbaserade priszoner
kan förbättra prognoser för Ethereums framtida
avkastning efter 6 och 12 månader.

#### Metod

Vi använde Expanding Window Cross-Validation
med kronologiskt uppdelade valideringsperioder.

Zonmodellen använde historisk medianavkastning
för respektive regressionszon.

Referensmodellen använde medianavkastningen
från samtliga historiska träningsobservationer.

Prognoserna utvärderades med Mean Absolute
Error (MAE). Lägre MAE innebär bättre resultat.

#### Resultat – 6 månader

| Modell | MAE |
|---|---:|
| Regressionszoner | 57,92 |
| Referensmodell | 54,06 |

Zonmodellen hade 3,86 procentenheter högre
genomsnittligt prognosfel än referensmodellen.

Zonmodellen presterade bättre under 2023 och
2024, men sämre under 2021 och 2022.

#### Resultat – 12 månader

| Modell | MAE |
|---|---:|
| Regressionszoner | 84,44 |
| Referensmodell | 62,84 |

Zonmodellen hade 21,60 procentenheter högre
genomsnittligt prognosfel än referensmodellen.

Under 2021 presterade zonmodellen bättre,
men under 2022 och 2023 betydligt sämre.

Valideringsåret 2024 saknar 12M-resultat
eftersom framtida utfall begränsades till
datum före 2025-01-01.

#### Analys

Resultaten visar att regressionszonernas
prognosförmåga varierar mellan marknadsperioder.

En möjlig förklaring är att sambandet mellan
historiska priszoner och framtida avkastning
förändras mellan olika marknadscykler.

Modellen använder dessutom endast tre
priszoner, vilket begränsar möjligheten
att beskriva variationer inom varje zon.

#### Metodologiska begränsningar

- Framtida avkastningsperioder överlappar.
- Observationerna är inte oberoende.
- Stora prisförändringar påverkar MAE.
- Zondefinitionerna har undersökts tidigare.
- Detta är inte ett oberoende sluttest.
- Zonreglerna har inte optimerats inom
  separata träningsfönster.

#### Slutsats

Den regressionsbaserade zonmodellen gav
inte bättre genomsnittlig prognosprecision
än referensmodellen för någon av horisonterna.

Resultaten visar att modellen kan beskriva
historiska priszoner, men ger inte tillräckligt
stöd för att zonerna ska kunna användas
för tillförlitliga avkastningsprognoser.

Ytterligare validering krävs för att bedöma
om regressionszonerna kan identifiera
framtida gynnsamma köpperioder.



## Steg 11 – 12-månadersprojektion för Ethereum

### Syfte

Vi undersöker hur Ethereums långsiktiga
regressionsmodell utvecklas vid en matematisk
projektion 12 månader framåt.

### Metod

Vi använder den senaste tillgängliga
Expanding Window Regression-modellen.

Projektionen omfattar 365 dagar.

Vi beräknar:
1. Den framtida regressionslinjen.
2. Övre och nedre regressionsband.
3. Skillnaden mot det senaste ETH-priset.
4. Historiska prognosfel för samma tidshorisont.

Regressionsbanden är modellbaserade
residualband och ska inte tolkas som
statistiskt kalibrerade prediktionsintervall.

### Begränsningar

Modellen beskriver historisk exponentiell
prisutveckling och kan överskatta framtida
prisnivåer om tillväxtmönstret förändras.

Projektionen är ett matematiskt scenario,
inte en verifierad marknadsprognos
eller investeringsrekommendation.


In [19]:

# STEG 11 – Logaritmisk regression 12 månader framåt

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

# Historiska ETH-priser
forecast_data = df_eth.copy()
forecast_data = (
    forecast_data.sort_values("time")
    .dropna(subset=["ETH_Price"])
    .reset_index(drop=True)
)

forecast_data = forecast_data[
    forecast_data["ETH_Price"] > 0
].reset_index(drop=True)

last_date = forecast_data["time"].iloc[-1]
last_price = float(forecast_data["ETH_Price"].iloc[-1])
first_date = forecast_data["time"].iloc[0]

# Historisk tid i dagar
x = (
    (forecast_data["time"] - first_date)
    .dt.total_seconds().to_numpy() / 86400
)

y = np.log(forecast_data["ETH_Price"].to_numpy())

# Regression med tillgängliga data till sista datum
slope, intercept = np.polyfit(x, y, 1)

fitted_log = intercept + slope * x

# Historisk residualspridning
sigma = np.std(y - fitted_log, ddof=2)

# 365 dagar framåt
future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=365,
    freq="D",
    tz="UTC"
)

future_x = (
    (future_dates - first_date)
    .total_seconds() / 86400
)

future_log = intercept + slope * future_x

# Matematiskt projicerad regressionslinje
future_trend = np.exp(future_log)

# Modellbaserade residualband
future_upper = np.exp(future_log + sigma)
future_lower = np.exp(future_log - sigma)

df_forecast = pd.DataFrame({
    "time": future_dates,
    "Regression_Trend": future_trend,
    "Lower_Band": future_lower,
    "Upper_Band": future_upper
})

# Tabell: sista projicerade datum
forecast_12m = float(df_forecast["Regression_Trend"].iloc[-1])

summary_forecast = pd.DataFrame({
    "Mått": [
        "Senaste ETH-pris (USD)",
        "Regressionsnivå idag (USD)",
        "Regressionsnivå efter 12M (USD)",
        "Nedre residualband efter 12M (USD)",
        "Övre residualband efter 12M (USD)",
        "Förändring från dagens ETH-pris (%)"
    ],
    "Värde": [
        last_price,
        np.exp(intercept + slope * x[-1]),
        forecast_12m,
        float(future_lower[-1]),
        float(future_upper[-1]),
        (forecast_12m / last_price - 1) * 100
    ]
})

print("Prognosdatum:", last_date)
print("Projektionens slutdatum:", future_dates[-1])
print("\n12-MÅNADERSPROJEKTION")

display(summary_forecast.round(2))


Prognosdatum: 2026-11-01 00:00:00+00:00
Projektionens slutdatum: 2027-11-01 00:00:00+00:00

12-MÅNADERSPROJEKTION


,Mått,Värde
0,Senaste ETH-pris (USD),2269.99
1,Regressionsnivå idag (USD),10892.01
2,Regressionsnivå efter 12M (USD),19117.25
3,Nedre residualband efter 12M (USD),5739.94
4,Övre residualband efter 12M (USD),63671.30
5,Förändring från dagens ETH-pris (%),742.17


In [20]:

# STEG 11.2 – Framtida regressionsband

fig_bands = go.Figure()

# Övre band
fig_bands.add_trace(go.Scatter(
    x=df_forecast["time"],
    y=df_forecast["Upper_Band"],
    name="Övre residualband",
    mode="lines",
    line=dict(color="#2ECC71", width=1)
))

# Nedre band med fyllning
fig_bands.add_trace(go.Scatter(
    x=df_forecast["time"],
    y=df_forecast["Lower_Band"],
    name="Nedre residualband",
    mode="lines",
    fill="tonexty",
    fillcolor="rgba(46,204,113,0.18)",
    line=dict(color="#2ECC71", width=1)
))

# Regressionslinje
fig_bands.add_trace(go.Scatter(
    x=df_forecast["time"],
    y=df_forecast["Regression_Trend"],
    name="Projicerad regressionsnivå",
    mode="lines",
    line=dict(color="#F39C12", width=3)
))

# Senaste kända ETH-pris – jämförelsenivå
fig_bands.add_hline(
    y=last_price,
    line_dash="dash",
    line_color="#5DADE2",
    annotation_text="Senaste ETH-pris"
)

fig_bands.update_layout(
    title="Ethereum – 12-månadersprojektion med regressionsband",
    xaxis_title="Datum",
    yaxis_title="ETH-pris (USD)",
    template="plotly_dark",
    height=600,
    hovermode="x unified"
)

fig_bands.show()


In [21]:

# STEG 11.3 – Historiska 12M-prognoser

historical_results = []

prices = forecast_data["ETH_Price"].to_numpy()
dates = forecast_data["time"]

# Minst ett års prishistorik
min_train = 365

# En prognos ungefär var 30:e dag
# 365 dagar framåt måste finnas i historiken
for i in range(
    min_train,
    len(forecast_data) - 365,
    30
):

    # Endast tidigare och dagens tillgängliga priser
    x_train = x[:i + 1]
    y_train = np.log(prices[:i + 1])

    slope_i, intercept_i = np.polyfit(
        x_train,
        y_train,
        1
    )

    # Förutsäg pris 365 dagar framåt
    prediction = np.exp(
        intercept_i + slope_i * (x[i] + 365)
    )

    actual = prices[i + 365]
    baseline = prices[i]

    historical_results.append({
        "Prognosdatum": dates.iloc[i],
        "Utfall_datum": dates.iloc[i + 365],
        "Faktiskt_pris": actual,
        "Regressionsprognos": prediction,
        "Referensprognos": baseline,
        "Fel_regression_pct": (
            abs(prediction - actual) / actual * 100
        ),
        "Fel_referens_pct": (
            abs(baseline - actual) / actual * 100
        )
    })

df_historical_forecasts = pd.DataFrame(
    historical_results
)

print("Antal historiska prognoser:",
      len(df_historical_forecasts))

historical_summary = pd.DataFrame({
    "Modell": [
        "Logaritmisk regression",
        "Oförändrat pris"
    ],
    "MAPE (%)": [
        df_historical_forecasts[
            "Fel_regression_pct"
        ].mean(),
        df_historical_forecasts[
            "Fel_referens_pct"
        ].mean()
    ]
})

print("\nHISTORISKT PROGNOSFEL")
display(historical_summary.round(2))

print("\nSenaste prognosjämförelserna:")
display(
    df_historical_forecasts.tail(10).round(2)
)


Antal historiska prognoser: 113

HISTORISKT PROGNOSFEL


,Modell,MAPE (%)
0,Logaritmisk regression,1724.56
1,Oförändrat pris,81.23



Senaste prognosjämförelserna:


C:\Users\milen\AppData\Local\Temp\ipykernel_10760\2649904496.py:78: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_historical_forecasts.tail(10).round(2)


,Prognosdatum,Utfall_datum,Faktiskt_pris,Regressionsprognos,Referensprognos,Fel_regression_pct,Fel_referens_pct
103,2025-01-22 00:00:00+00:00,2026-01-22 00:00:00+00:00,2947.89,18688.23,3239.68,533.95,9.90
104,2025-02-21 00:00:00+00:00,2026-02-21 00:00:00+00:00,1974.54,18877.33,2655.74,856.04,34.50
105,2025-03-23 00:00:00+00:00,2026-03-23 00:00:00+00:00,2148.13,18836.07,1997.88,776.86,6.99
106,2025-04-22 00:00:00+00:00,2026-04-22 00:00:00+00:00,2381.75,18630.82,1758.00,682.23,26.19
107,2025-05-22 00:00:00+00:00,2026-05-22 00:00:00+00:00,2066.77,18582.22,2657.42,799.10,28.58
108,2025-06-21 00:00:00+00:00,2026-06-21 00:00:00+00:00,1706.92,18657.97,2266.04,993.08,32.76
109,2025-07-21 00:00:00+00:00,2026-07-21 00:00:00+00:00,1923.18,18783.77,3765.90,876.71,95.82
110,2025-08-20 00:00:00+00:00,2026-08-20 00:00:00+00:00,2326.67,19153.24,4337.66,723.20,86.43
111,2025-09-19 00:00:00+00:00,2026-09-19 00:00:00+00:00,2632.92,19597.37,4463.60,644.32,69.53
112,2025-10-19 00:00:00+00:00,2026-10-19 00:00:00+00:00,2405.71,19984.61,3988.24,730.72,65.78


In [22]:

# STEG 11.4 – Visualisera historiska prognosfel

import plotly.graph_objects as go

fig_history = go.Figure()

fig_history.add_trace(go.Scatter(
    x=df_historical_forecasts["Utfall_datum"],
    y=df_historical_forecasts["Faktiskt_pris"],
    name="Faktiskt ETH-pris",
    mode="lines+markers",
    line=dict(color="#5DADE2", width=2)
))

fig_history.add_trace(go.Scatter(
    x=df_historical_forecasts["Utfall_datum"],
    y=df_historical_forecasts["Regressionsprognos"],
    name="Logaritmisk regressionsprognos",
    mode="lines+markers",
    line=dict(color="#F39C12", width=2)
))

fig_history.add_trace(go.Scatter(
    x=df_historical_forecasts["Utfall_datum"],
    y=df_historical_forecasts["Referensprognos"],
    name="Oförändrat pris – referens",
    mode="lines",
    line=dict(color="#2ECC71", dash="dash")
))

fig_history.update_layout(
    title="ETH – Historiska 12M-prognoser jämfört med verkligt pris",
    xaxis_title="Utfallsdatum",
    yaxis_title="ETH-pris (USD, logaritmisk skala)",
    yaxis_type="log",
    template="plotly_dark",
    height=650,
    hovermode="x unified"
)

fig_history.update_xaxes(rangeslider_visible=True)

fig_history.show()



### Resultat och slutsats – Steg 11

#### Syfte

Vi har undersökt hur logaritmisk regression
kan användas för att skapa en matematisk
12-månadersprojektion för Ethereum.

#### Metod

Vi använde historiska ETH-priser från 2015
fram till den senaste tillgängliga observationen.

En logaritmisk regressionsmodell anpassades
till prishistoriken och extrapolerades
365 dagar framåt.

Vi beräknade även övre och nedre residualband.

Därefter utvärderade vi historiska
12-månadersprognoser och jämförde dem med
en referensmodell som antar oförändrat pris.

#### Resultat

Resultaten redovisas i tabellerna och
de interaktiva diagrammen ovan.

Vi analyserar:

- Projicerad regressionsnivå efter 12 månader.
- Avvikelse från det senaste ETH-priset.
- Spridning mellan regressionsbanden.
- Historiskt prognosfel (MAPE).
- Jämförelse med referensmodell.

#### Metodbegränsningar

Modellen bygger på historisk exponentiell
prisutveckling och kan ge missvisande
framtida nivåer vid strukturella
förändringar på kryptomarknaden.

Residualbanden är inte statistiskt
kalibrerade prediktionsintervall.

Historiska prognoser använder endast
information som fanns tillgänglig
vid varje prognostillfälle.

Prognosernas utfallsperioder överlappar
och resultaten är därmed inte oberoende.

#### Slutsats

Den matematiska regressionsprojektionen
måste bedömas tillsammans med resultaten
från den historiska prognosutvärderingen.

En framtida regressionsnivå ska inte
tolkas som ett förväntat marknadspris
utan stöd från oberoende validering.

Den slutliga slutsatsen formuleras
efter granskning av resultaten.



### Resultat och slutsats – Steg 11

#### Syfte

Vi undersökte om logaritmisk regression
kan användas för att skapa en 12-månadersprojektion
av Ethereums framtida prisutveckling.

Vi utvärderade även modellens historiska
prognosprecision genom att jämföra tidigare
prognoser med faktiskt observerade priser.

#### Metod

Vi använde ETH-priser från augusti 2015
till november 2026.

En logaritmisk regressionsmodell anpassades
till historiska priser och extrapolerades
365 dagar framåt.

Vi skapade även övre och nedre residualband.

För att utvärdera modellen genomfördes
113 historiska prognosexperiment med
Expanding Window Regression.

Regressionsprognoserna jämfördes med
en referensmodell som antar oförändrat pris.

#### Resultat – 12-månadersprojektion

| Mått | Resultat |
|---|---:|
| Senaste ETH-pris | 2 269,99 USD |
| Regressionsnivå idag | 10 892,01 USD |
| Regressionsnivå efter 12M | 19 117,25 USD |
| Nedre residualband | 5 739,94 USD |
| Övre residualband | 63 671,30 USD |
| Skillnad mot senaste pris | +742,17 % |

Projektionens slutdatum är 2027-11-01.

Den matematiska modellen visar en kraftigt
stigande långsiktig regressionsnivå.

Samtidigt är skillnaden mellan den historiska
regressionsnivån och det faktiska ETH-priset
mycket stor.

Det indikerar att modellen inte längre
beskriver de senaste prisnivåerna särskilt väl.

#### Resultat – Historisk prognosprecision

Totalt utvärderades 113 historiska prognoser.

| Modell | MAPE |
|---|---:|
| Logaritmisk regression | 1 724,56 % |
| Referensmodell: Oförändrat pris | 81,23 % |

Den logaritmiska regressionsmodellen hade
betydligt högre prognosfel än referensmodellen.

De historiska diagrammen visar att
regressionsmodellen ofta överskattade
Ethereums framtida prisnivåer.

#### Analys

Modellen bygger på antagandet att Ethereums
logaritmerade pris kan beskrivas av en
långsiktig linjär trend.

Detta motsvarar en exponentiell utveckling
av priset över tid.

Kryptomarknaden påverkas dock av förändrade
marknadsförhållanden, efterfrågan, utbud
och andra ekonomiska faktorer.

Därför behöver historisk exponentiell
pristillväxt inte fortsätta i framtiden.

Modellen riskerar att extrapolera tidigare
prisökningar och därmed skapa alltför
höga framtida prisprojektioner.

#### Metodologiska begränsningar

- Historiska prognoser överlappar i tiden.
- MAPE påverkas kraftigt av stora prognosfel.
- Referensmodellen är enkel men användbar.
- Regressionsbanden är inte kalibrerade
  prediktionsintervall.
- Projektionen tar inte hänsyn till
  framtida förändringar i marknaden.
- Modellen är inte verifierad för
  praktisk handel eller investeringsbeslut.

#### Slutsats

Den logaritmiska regressionsmodellen
visar begränsad förmåga att förutsäga
Ethereums pris 12 månader framåt.

Den historiska utvärderingen visar att
referensmodellen med oförändrat pris
hade betydligt lägre prognosfel.

Den projicerade regressionsnivån
på 19 117 USD ska därför inte tolkas
som en tillförlitlig framtida prisprognos.

Resultaten visar att logaritmisk regression
kan vara användbar för att studera
historiska priszoner, men att modellen
inte är tillräckligt robust för att ensam
användas som en framtida prisprognos.

Ytterligare modeller och oberoende
validering behövs för att undersöka
om prognosprecisionen kan förbättras.



## Steg 12 – Jämförelse av regressionsmodeller och Machine Learning

### Forskningsfråga

Kan mer flexibla regressionsmodeller och
maskininlärning förbättra identifieringen
av historiskt gynnsamma köpzoner för Ethereum?

### Modeller

1. Expanding Window Regression
2. Rolling Window Regression (730 dagar)
3. Random Forest Classifier

### Prognoshorisonter

- 6 månader: 180 dagar
- 12 månader: 365 dagar

### Utvärderingsmetod

Vi använder kronologisk Expanding Window
Cross-Validation.

För varje valideringsperiod använder vi
endast träningsutfall som redan hunnit bli kända.

Vi jämför modellernas sannolikhetsprognoser
med hjälp av Brier Score.

Lägre Brier Score innebär bättre resultat.

Vi använder även ROC-AUC när båda
utfallsklasserna finns i testperioden.

Resultaten utvärderas på samma datum
för samtliga tre modeller.

### Begränsningar

En positiv framtida avkastning är inte
nödvändigtvis detsamma som en gynnsam
köpzon jämfört med andra investeringsalternativ.

Modellerna utvärderas därför först som
klassificeringsmodeller. Därefter behöver
deras användbarhet för investeringar
bedömas separat genom backtesting.


In [23]:

# STEG 12.1 – Rolling Regression för Ethereum

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

rolling_df = df_eth.copy()
rolling_df = rolling_df.sort_values(
    "time"
).reset_index(drop=True)

prices = rolling_df["ETH_Price"].to_numpy()
log_prices = np.log(prices)

x = (
    (rolling_df["time"] - rolling_df["time"].iloc[0])
    .dt.total_seconds().to_numpy() / 86400
)

WINDOW = 730

rolling_trend = np.full(len(rolling_df), np.nan)
rolling_upper = np.full(len(rolling_df), np.nan)
rolling_lower = np.full(len(rolling_df), np.nan)

for i in range(WINDOW, len(rolling_df)):

    # Endast tidigare observationer används
    x_train = x[i-WINDOW:i]
    y_train = log_prices[i-WINDOW:i]

    slope, intercept = np.polyfit(
        x_train, y_train, 1
    )

    fitted = intercept + slope * x_train

    sigma = np.std(
        y_train - fitted,
        ddof=2
    )

    prediction = intercept + slope * x[i]

    rolling_trend[i] = np.exp(prediction)
    rolling_upper[i] = np.exp(prediction + sigma)
    rolling_lower[i] = np.exp(prediction - sigma)

rolling_df["Rolling_Trend"] = rolling_trend
rolling_df["Rolling_Upper"] = rolling_upper
rolling_df["Rolling_Lower"] = rolling_lower

# Klassificera zoner
rolling_df["Rolling_Zone"] = "INGEN DATA"

valid = rolling_df["Rolling_Trend"].notna()

rolling_df.loc[valid, "Rolling_Zone"] = "Neutral zon"

rolling_df.loc[
    valid &
    (rolling_df["ETH_Price"] < rolling_df["Rolling_Lower"]),
    "Rolling_Zone"
] = "Nedre zon"

rolling_df.loc[
    valid &
    (rolling_df["ETH_Price"] > rolling_df["Rolling_Upper"]),
    "Rolling_Zone"
] = "Övre zon"

print("ROLLING REGRESSION – RESULTAT")

print("\nFörsta giltiga datum:")
print(
    rolling_df.loc[
        valid, "time"
    ].min()
)

print("\nFördelning av priszoner:")
display(
    rolling_df["Rolling_Zone"]
    .value_counts()
    .rename_axis("Priszon")
    .reset_index(name="Antal dagar")
)

print("\nSenaste observationer:")
display(
    rolling_df[
        [
            "time",
            "ETH_Price",
            "Rolling_Trend",
            "Rolling_Lower",
            "Rolling_Upper",
            "Rolling_Zone"
        ]
    ].tail(10).round(2)
)


ROLLING REGRESSION – RESULTAT

Första giltiga datum:
2017-08-07 00:00:00+00:00

Fördelning av priszoner:


,Priszon,Antal dagar
0,Neutral zon,1568
1,Nedre zon,978
2,Övre zon,828
3,INGEN DATA,730



Senaste observationer:


C:\Users\milen\AppData\Local\Temp\ipykernel_10760\3209641455.py:101: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,ETH_Price,Rolling_Trend,Rolling_Lower,Rolling_Upper,Rolling_Zone
4094,2026-10-23 00:00:00+00:00,2378.26,2207.29,1686.86,2888.29,Neutral zon
4095,2026-10-24 00:00:00+00:00,2351.99,2205.78,1685.89,2885.99,Neutral zon
4096,2026-10-25 00:00:00+00:00,2325.80,2204.16,1684.84,2883.55,Neutral zon
4097,2026-10-26 00:00:00+00:00,2316.60,2202.12,1683.58,2880.39,Neutral zon
4098,2026-10-27 00:00:00+00:00,2347.56,2200.20,1682.35,2877.47,Neutral zon
4099,2026-10-28 00:00:00+00:00,2356.99,2198.51,1681.26,2874.90,Neutral zon
4100,2026-10-29 00:00:00+00:00,2256.03,2197.01,1680.28,2872.65,Neutral zon
4101,2026-10-30 00:00:00+00:00,2240.23,2195.15,1678.99,2869.98,Neutral zon
4102,2026-10-31 00:00:00+00:00,2252.07,2193.26,1677.67,2867.30,Neutral zon
4103,2026-11-01 00:00:00+00:00,2269.99,2191.11,1676.25,2864.11,Neutral zon


In [24]:

# STEG 12.2 – Interaktiva regressionsdiagram

# DIAGRAM 1 – Rolling Regression
fig_roll = go.Figure()

fig_roll.add_trace(go.Scatter(
    x=rolling_df["time"],
    y=rolling_df["Rolling_Upper"],
    name="Övre rolling-band",
    line=dict(color="#2ECC71", width=1)
))

fig_roll.add_trace(go.Scatter(
    x=rolling_df["time"],
    y=rolling_df["Rolling_Lower"],
    name="Nedre rolling-band",
    fill="tonexty",
    fillcolor="rgba(46,204,113,0.16)",
    line=dict(color="#2ECC71", width=1)
))

fig_roll.add_trace(go.Scatter(
    x=rolling_df["time"],
    y=rolling_df["Rolling_Trend"],
    name="Rolling Regression",
    line=dict(color="#F39C12", width=2)
))

fig_roll.add_trace(go.Scatter(
    x=rolling_df["time"],
    y=rolling_df["ETH_Price"],
    name="ETH-pris",
    line=dict(color="#5DADE2", width=1.5)
))

fig_roll.update_layout(
    title="Ethereum – Rolling Regression (730 dagar)",
    xaxis_title="Datum",
    yaxis_title="ETH-pris (USD, logaritmisk skala)",
    yaxis_type="log",
    template="plotly_dark",
    height=650,
    hovermode="x unified"
)

fig_roll.update_xaxes(rangeslider_visible=True)
fig_roll.show()


# DIAGRAM 2 – Jämförelse av regressionstrender
fig_compare = go.Figure()

fig_compare.add_trace(go.Scatter(
    x=rolling_df["time"],
    y=rolling_df["ETH_Price"],
    name="ETH-pris",
    line=dict(color="#5DADE2")
))

fig_compare.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["Regression_Trend"],
    name="Expanding Regression",
    line=dict(color="#FF4B4B", width=2)
))

fig_compare.add_trace(go.Scatter(
    x=rolling_df["time"],
    y=rolling_df["Rolling_Trend"],
    name="Rolling Regression",
    line=dict(color="#2ECC71", width=2)
))

fig_compare.update_layout(
    title="Ethereum – Expanding vs Rolling Regression",
    xaxis_title="Datum",
    yaxis_title="ETH-pris (USD, logaritmisk skala)",
    yaxis_type="log",
    template="plotly_dark",
    height=650,
    hovermode="x unified"
)

fig_compare.update_xaxes(rangeslider_visible=True)
fig_compare.show()


In [25]:

# STEG 12.3 – Förbered data för Machine Learning

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score
import numpy as np
import pandas as pd

# Utgå från befintlig avkastningsanalys
ml = df_returns.copy()

# BTC-data från steg 1
btc_features = df_btc[
    ["time", "BTC_Price"]
].copy()

# Rullande regression
rolling_features = rolling_df[
    [
        "time",
        "Rolling_Trend",
        "Rolling_Upper",
        "Rolling_Lower",
        "Rolling_Zone"
    ]
].copy()

ml = ml.merge(
    btc_features,
    on="time",
    how="left",
    validate="one_to_one"
)

ml = ml.merge(
    rolling_features,
    on="time",
    how="left",
    validate="one_to_one"
)

ml = ml.sort_values("time").reset_index(drop=True)

# Teknisk analys – endast dåvarande och tidigare priser
ml["ETH_Return_30D"] = (
    ml["ETH_Price"].pct_change(30) * 100
)

ml["BTC_Return_30D"] = (
    ml["BTC_Price"].pct_change(30) * 100
)

ml["ETH_Return_Daily"] = (
    ml["ETH_Price"].pct_change() * 100
)

ml["ETH_Volatility_30D"] = (
    ml["ETH_Return_Daily"]
    .rolling(30)
    .std()
)

# RSI 14
delta = ml["ETH_Price"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.ewm(
    alpha=1/14,
    adjust=False,
    min_periods=14
).mean()

avg_loss = loss.ewm(
    alpha=1/14,
    adjust=False,
    min_periods=14
).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)

ml["RSI_14"] = 100 - (100 / (1 + rs))

# Relativa avstånd från regressionslinjer
ml["Expanding_Distance"] = (
    np.log(
        ml["ETH_Price"] / ml["Regression_Trend"]
    )
)

ml["Rolling_Distance"] = (
    np.log(
        ml["ETH_Price"] / ml["Rolling_Trend"]
    )
)

# Klassificera Rolling Regression-zoner
ml["Expanding_Zone"] = ml["Price_Zone"]

feature_cols = [
    "ETH_Return_30D",
    "BTC_Return_30D",
    "ETH_Volatility_30D",
    "RSI_14",
    "Expanding_Distance",
    "Rolling_Distance"
]

# Kontrollera saknade värden
print("Saknade featurevärden:")
display(ml[feature_cols].isna().sum())

# Behåll samma giltiga featuredatum
ml = ml.dropna(subset=feature_cols).copy()

ml = ml[
    (ml["Expanding_Zone"] != "INGEN DATA")
    & (ml["Rolling_Zone"] != "INGEN DATA")
].copy()

print("\nObservationer för modelljämförelse:", len(ml))
print("Första datum:", ml["time"].min())
print("Sista datum:", ml["time"].max())

display(
    ml[
        ["time"] + feature_cols
    ].tail(5).round(3)
)


Saknade featurevärden:


ETH_Return_30D         30
BTC_Return_30D         30
ETH_Volatility_30D     30
RSI_14                 14
Expanding_Distance      0
Rolling_Distance      365
dtype: int64


Observationer för modelljämförelse: 3374
Första datum: 2017-08-07 00:00:00+00:00
Sista datum: 2026-11-01 00:00:00+00:00


C:\Users\milen\AppData\Local\Temp\ipykernel_10760\1516498140.py:128: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(5).round(3)


,time,ETH_Return_30D,BTC_Return_30D,ETH_Volatility_30D,RSI_14,Expanding_Distance,Rolling_Distance
3734,2026-10-28 00:00:00+00:00,-12.348,-2.729,1.751,39.319,-1.532,0.070
3735,2026-10-29 00:00:00+00:00,-15.777,-6.551,1.887,31.483,-1.576,0.027
3736,2026-10-30 00:00:00+00:00,-16.598,-7.741,1.881,30.460,-1.583,0.020
3737,2026-10-31 00:00:00+00:00,-16.701,-8.701,1.878,32.238,-1.578,0.026
3738,2026-11-01 00:00:00+00:00,-14.918,-7.552,1.890,34.947,-1.570,0.035


In [26]:

# STEG 12.4 – Gemensam tidsbaserad modelljämförelse

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score

validation_years = [2021, 2022, 2023, 2024]
horizons = {"6M": 180, "12M": 365}

evaluation_results = []
all_model_predictions = []

for horizon, days in horizons.items():

    target_col = f"Return_{horizon}"

    data = ml.copy()

    # Positiv ETH-avkastning som målvariabel
    data["Target"] = (
        data[target_col] > 0
    ).astype(int)

    # Datum när framtida avkastning är känd
    data["Exit_Date"] = (
        data["time"] + pd.Timedelta(days=days + 1)
    )

    # Saknade framtida utfall får inte ingå
    data = data.dropna(subset=[target_col]).copy()

    for year in validation_years:

        start = pd.Timestamp(
            f"{year}-01-01", tz="UTC"
        )

        end = pd.Timestamp(
            f"{year+1}-01-01", tz="UTC"
        )

        # Träning: endast redan kända utfall
        train = data[
            data["Exit_Date"] < start
        ].copy()

        # Validering: det aktuella året
        validation = data[
            (data["time"] >= start)
            & (data["time"] < end)
        ].copy()

        if len(train) < 100 or validation.empty:
            continue

        if train["Target"].nunique() < 2:
            continue

        # Baseline: historisk andel positiva utfall
        baseline_prob = train["Target"].mean()

        # Modell 1 – Expanding Regression-zoner
        exp_prob = (
            train.groupby("Expanding_Zone")["Target"]
            .mean()
        )

        validation["Expanding_Prob"] = (
            validation["Expanding_Zone"]
            .map(exp_prob)
            .fillna(baseline_prob)
        )

        # Modell 2 – Rolling Regression-zoner
        roll_prob = (
            train.groupby("Rolling_Zone")["Target"]
            .mean()
        )

        validation["Rolling_Prob"] = (
            validation["Rolling_Zone"]
            .map(roll_prob)
            .fillna(baseline_prob)
        )

        # Modell 3 – Random Forest
        rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(
            train[feature_cols],
            train["Target"]
        )

        positive_index = list(
            rf.classes_
        ).index(1)

        validation["RF_Prob"] = (
            rf.predict_proba(
                validation[feature_cols]
            )[:, positive_index]
        )

        validation["Baseline_Prob"] = baseline_prob

        validation["Horisont"] = horizon
        validation["Valideringsår"] = year

        model_columns = {
            "Baseline": "Baseline_Prob",
            "Expanding Regression": "Expanding_Prob",
            "Rolling Regression": "Rolling_Prob",
            "Random Forest": "RF_Prob"
        }

        for model_name, prob_col in model_columns.items():

            actual = validation["Target"]
            predicted = validation[prob_col]

            brier = brier_score_loss(
                actual, predicted
            )

            # ROC-AUC kräver båda klasserna
            auc = np.nan

            if actual.nunique() == 2:
                auc = roc_auc_score(
                    actual, predicted
                )

            evaluation_results.append({
                "Horisont": horizon,
                "År": year,
                "Modell": model_name,
                "Observationer": len(validation),
                "Brier Score": brier,
                "ROC-AUC": auc
            })

        all_model_predictions.append(
            validation[
                [
                    "time",
                    "Horisont",
                    "Valideringsår",
                    "Target",
                    target_col,
                    "Expanding_Zone",
                    "Rolling_Zone",
                    "Baseline_Prob",
                    "Expanding_Prob",
                    "Rolling_Prob",
                    "RF_Prob"
                ]
            ].copy()
        )

# Samla resultaten
df_model_results = pd.DataFrame(
    evaluation_results
)

df_model_predictions = pd.concat(
    all_model_predictions,
    ignore_index=True
)

print("MODELLJÄMFÖRELSE – PER ÅR")

display(
    df_model_results.round(4)
)

# Samlat Brier Score per modell och horisont
summary_rows = []

model_columns = {
    "Baseline": "Baseline_Prob",
    "Expanding Regression": "Expanding_Prob",
    "Rolling Regression": "Rolling_Prob",
    "Random Forest": "RF_Prob"
}

for horizon, group in df_model_predictions.groupby("Horisont"):

    for model_name, column in model_columns.items():

        summary_rows.append({
            "Horisont": horizon,
            "Modell": model_name,
            "Observationer": len(group),
            "Brier Score": brier_score_loss(
                group["Target"],
                group[column]
            )
        })

df_model_summary = pd.DataFrame(summary_rows)

print("\nSAMMANFATTNING – ALLA MODELLER")

display(
    df_model_summary.sort_values(
        ["Horisont", "Brier Score"]
    ).round(4)
)


MODELLJÄMFÖRELSE – PER ÅR


,Horisont,År,Modell,Observationer,Brier Score,ROC-AUC
0,6M,2021,Baseline,365,0.2396,0.5000
1,6M,2021,Expanding Regression,365,0.2495,0.5000
2,6M,2021,Rolling Regression,365,0.1945,0.8759
3,6M,2021,Random Forest,365,0.3709,0.2323
4,6M,2022,Baseline,365,0.2973,0.5000
5,6M,2022,Expanding Regression,365,0.4092,0.0645
6,6M,2022,Rolling Regression,365,0.2277,0.6067
7,6M,2022,Random Forest,365,0.2856,0.5169
8,6M,2023,Baseline,365,0.2084,0.5000
9,6M,2023,Expanding Regression,365,0.1982,0.7471



SAMMANFATTNING – ALLA MODELLER


,Horisont,Modell,Observationer,Brier Score
1,12M,Expanding Regression,1461,0.2292
3,12M,Random Forest,1461,0.2426
0,12M,Baseline,1461,0.2427
2,12M,Rolling Regression,1461,0.2873
6,6M,Rolling Regression,1461,0.2374
4,6M,Baseline,1461,0.2617
5,6M,Expanding Regression,1461,0.2867
7,6M,Random Forest,1461,0.3052


In [27]:

# STEG 12.5 – Interaktiva diagram för modelljämförelse

import plotly.express as px
import plotly.graph_objects as go

# DIAGRAM 1 – Brier Score för alla modeller
fig_brier = px.bar(
    df_model_summary,
    x="Modell",
    y="Brier Score",
    color="Modell",
    facet_col="Horisont",
    text_auto=".4f",
    title="ETH – Modelljämförelse: Brier Score",
    template="plotly_dark"
)

fig_brier.update_layout(
    height=550,
    showlegend=False,
    yaxis_title="Brier Score (lägre är bättre)"
)

fig_brier.show()


# DIAGRAM 2 – Modellernas resultat per år
fig_yearly = px.line(
    df_model_results,
    x="År",
    y="Brier Score",
    color="Modell",
    facet_row="Horisont",
    markers=True,
    title="ETH – Prognosfel per valideringsår",
    template="plotly_dark"
)

fig_yearly.update_xaxes(type="category")
fig_yearly.update_layout(height=750)

fig_yearly.show()


# DIAGRAM 3 – Random Forest sannolikheter över tid
for horizon in ["6M", "12M"]:

    subset = df_model_predictions[
        df_model_predictions["Horisont"] == horizon
    ].sort_values("time")

    fig_prob = go.Figure()

    fig_prob.add_trace(go.Scatter(
        x=subset["time"],
        y=subset["RF_Prob"] * 100,
        name="Random Forest",
        mode="lines",
        line=dict(color="#2ECC71", width=1.5)
    ))

    fig_prob.add_trace(go.Scatter(
        x=subset["time"],
        y=subset["Baseline_Prob"] * 100,
        name="Referensmodell",
        mode="lines",
        line=dict(color="#F39C12", dash="dash")
    ))

    fig_prob.add_hline(
        y=50,
        line_dash="dot",
        line_color="gray"
    )

    fig_prob.update_layout(
        title=(
            "Ethereum – Prognostiserad sannolikhet "
            f"för positiv avkastning ({horizon})"
        ),
        xaxis_title="Datum",
        yaxis_title="Sannolikhet (%)",
        yaxis_range=[0, 100],
        template="plotly_dark",
        height=550
    )

    fig_prob.update_xaxes(rangeslider_visible=True)
    fig_prob.show()



### Resultat och slutsats – Steg 12

#### Syfte

Syftet var att undersöka om regressionsbaserade
modeller och maskininlärning kan identifiera
historiskt gynnsamma köpperioder för Ethereum.

Vi jämförde tre modeller med en enkel referensmodell.

#### Modeller

1. Baseline – historisk andel positiva utfall.
2. Expanding Window Regression.
3. Rolling Window Regression (730 dagar).
4. Random Forest Classifier.

Random Forest använde historiska indikatorer
från Ethereum och Bitcoin, bland annat
prisförändringar, volatilitet, RSI och
avstånd till regressionslinjer.

#### Metod

Vi använde kronologisk Expanding Window
Cross-Validation för åren 2021–2024.

Prognoshorisonterna var:

- 6 månader (180 dagar)
- 12 månader (365 dagar)

Vi utvärderade sannolikheten för positiv
framtida avkastning med Brier Score.

ROC-AUC beräknades när båda utfallsklasserna
förekom i valideringsperioden.

#### Resultat – 6 månader

| Modell | Brier Score |
|---|---:|
| Baseline | 0,2617 |
| Expanding Regression | 0,2867 |
| Rolling Regression | 0,2374 |
| Random Forest | 0,3052 |

Rolling Regression hade lägst Brier Score.

Detta innebär att Rolling Regression gav
bäst sammanlagd sannolikhetsprognos för
positiv avkastning efter 6 månader.

Random Forest hade högst prognosfel.

#### Resultat – 12 månader

| Modell | Brier Score |
|---|---:|
| Baseline | 0,2427 |
| Expanding Regression | 0,2292 |
| Rolling Regression | 0,2873 |
| Random Forest | 0,2426 |

Expanding Regression hade lägst Brier Score
för 12-månadershorisonten.

Random Forest presterade ungefär på samma
nivå som referensmodellen.

Rolling Regression hade högst prognosfel.

#### Analys

Resultaten visar att olika modeller fungerar
olika bra beroende på prognoshorisonten.

Rolling Regression anpassar sig till de
senaste två årens prisutveckling och gav
lägst prognosfel för 6 månader.

Expanding Regression använder en växande
historisk datamängd och gav lägst
prognosfel för 12 månader.

Random Forest identifierade varierande
sannolikheter för framtida prisuppgång,
men gav inte bättre sammanlagda resultat
än referensmodellen.

Det visar att mer komplexa ML-modeller
inte nödvändigtvis ger bättre prognoser.

#### Metodologiska begränsningar

- Valideringsperioderna innehåller
  överlappande framtida avkastningsutfall.
- Resultaten varierar mellan marknadsår.
- ROC-AUC kan inte beräknas när endast
  en utfallsklass förekommer.
- Positiv framtida avkastning är inte
  samma sak som en optimal köpzon.
- Hyperparametrarna har inte optimerats
  inom en separat modellvalsprocess.
- Den samlade analysen är explorativ
  och inte ett oberoende sluttest.

#### Slutsats

Rolling Regression gav bäst sammanlagd
prognosprecision för 6 månader.

Expanding Regression gav bäst sammanlagd
prognosprecision för 12 månader.

Random Forest överträffade inte
referensmodellen i den samlade analysen.

Resultaten tyder på att regressionsbaserade
modeller kan innehålla information om
framtida positiv avkastning.

Vi kan däremot inte fastställa att
modellerna identifierar lönsamma köpzoner
eller att resultaten är stabila på nya data.

Nästa steg är att utvärdera modellerna
på senare tidsperioder och undersöka
framtida avkastning samt investeringsrisk
för identifierade köpzoner.



## Steg 13 – Historisk testutvärdering 2025–2026

### Syfte

Vi undersöker om våra modeller kan identifiera
positiv framtida ETH-avkastning på senare
historiska observationer.

### Modeller

1. Baseline
2. Expanding Window Regression
3. Rolling Window Regression
4. Random Forest Classifier

### Metod

Modellerna tränas med observationer vars
framtida utfall var kända före 2025-01-01.

Vi använder en fast träningsgräns och
uppdaterar inte modellerna under testperioden.

Testperioden börjar 2025-01-01.

Prognoshorisonter:
- 6 månader (180 dagar)
- 12 månader (365 dagar)

Vi utvärderar:
- Brier Score
- ROC-AUC
- Andel positiva faktiska utfall
- Modellernas sannolikhetsprognoser

### Begränsningar

Utfallet måste finnas observerat i datasetet.

Senare observationer utan komplett framtida
avkastning exkluderas.

Testperioden har tidigare använts i delar
av projektets explorativa analyser och
är därför inte ett helt oberoende sluttest.



In [28]:

# STEG 13.1 – Historisk testutvärdering 2025–2026

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score
from IPython.display import display

TEST_START = pd.Timestamp("2025-01-01", tz="UTC")
HORIZONS = {"6M": 180, "12M": 365}

# Använd samma features som steg 12
test_source = ml.copy()
test_source["time"] = pd.to_datetime(
    test_source["time"], utc=True
)

test_source = test_source.sort_values(
    "time"
).reset_index(drop=True)

LAST_DATE = test_source["time"].max()

test_results = []
test_predictions = []

for horizon, days in HORIZONS.items():

    return_col = f"Return_{horizon}"

    data = test_source.copy()

    # Köpsimulering enligt steg 5:
    # köp t+1, utfall t+1+horizon
    data["Entry_Date"] = (
        data["time"] + pd.Timedelta(days=1)
    )

    data["Exit_Date"] = (
        data["Entry_Date"] + pd.Timedelta(days=days)
    )

    # Endast faktiskt kända utfall
    data = data[
        data[return_col].notna()
        & (data["Exit_Date"] <= LAST_DATE)
    ].copy()

    data["Target"] = (
        data[return_col] > 0
    ).astype(int)

    # Träna bara med historiskt kända utfall
    train = data[
        data["Exit_Date"] < TEST_START
    ].copy()

    # Testa senare observationer
    test = data[
        data["time"] >= TEST_START
    ].copy()

    if train.empty or test.empty:
        print(f"Inga tillräckliga data för {horizon}")
        continue

    if train["Target"].nunique() < 2:
        print(f"Träningsdata saknar två klasser: {horizon}")
        continue

    # Referens: historisk andel positiva utfall
    baseline_prob = train["Target"].mean()

    # Expanding Regression
    exp_rates = train.groupby(
        "Expanding_Zone"
    )["Target"].mean()

    test["Expanding_Prob"] = (
        test["Expanding_Zone"]
        .map(exp_rates)
        .fillna(baseline_prob)
    )

    # Rolling Regression
    rolling_rates = train.groupby(
        "Rolling_Zone"
    )["Target"].mean()

    test["Rolling_Prob"] = (
        test["Rolling_Zone"]
        .map(rolling_rates)
        .fillna(baseline_prob)
    )

    # Random Forest – samma inställningar som steg 12
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(
        train[feature_cols],
        train["Target"]
    )

    positive_class_index = list(
        rf.classes_
    ).index(1)

    test["RF_Prob"] = rf.predict_proba(
        test[feature_cols]
    )[:, positive_class_index]

    test["Baseline_Prob"] = baseline_prob
    test["Horisont"] = horizon

    models = {
        "Baseline": "Baseline_Prob",
        "Expanding Regression": "Expanding_Prob",
        "Rolling Regression": "Rolling_Prob",
        "Random Forest": "RF_Prob"
    }

    print(f"\n===== {horizon} =====")
    print("Träningsobservationer:", len(train))
    print("Testobservationer:", len(test))
    print("Första testdatum:", test["time"].min())
    print("Sista testdatum:", test["time"].max())
    print(
        "Andel positiva utfall:",
        round(test["Target"].mean() * 100, 2), "%"
    )

    for model_name, prob_col in models.items():

        brier = brier_score_loss(
            test["Target"],
            test[prob_col]
        )

        auc = np.nan

        if test["Target"].nunique() == 2:
            auc = roc_auc_score(
                test["Target"],
                test[prob_col]
            )

        test_results.append({
            "Horisont": horizon,
            "Modell": model_name,
            "Träning": len(train),
            "Test": len(test),
            "Brier Score": brier,
            "ROC-AUC": auc,
            "Positiva utfall (%)": (
                test["Target"].mean() * 100
            )
        })

    test_predictions.append(
        test[
            [
                "time",
                "Entry_Date",
                "Exit_Date",
                "Horisont",
                "ETH_Price",
                "Price_Zone",
                "Rolling_Zone",
                return_col,
                "Target",
                "Baseline_Prob",
                "Expanding_Prob",
                "Rolling_Prob",
                "RF_Prob"
            ]
        ].rename(columns={
            return_col: "Actual_Return"
        }).copy()
    )

# Sammanställ resultaten
df_test13 = pd.DataFrame(test_results)

df_predictions13 = pd.concat(
    test_predictions,
    ignore_index=True
)

print("\nRESULTAT – HISTORISK TESTPERIOD")

display(
    df_test13.sort_values(
        ["Horisont", "Brier Score"]
    ).round(4)
)



===== 6M =====
Träningsobservationer: 2523
Testobservationer: 489
Första testdatum: 2025-01-01 00:00:00+00:00
Sista testdatum: 2026-05-04 00:00:00+00:00
Andel positiva utfall: 50.92 %

===== 12M =====
Träningsobservationer: 2338
Testobservationer: 304
Första testdatum: 2025-01-01 00:00:00+00:00
Sista testdatum: 2025-10-31 00:00:00+00:00
Andel positiva utfall: 20.07 %

RESULTAT – HISTORISK TESTPERIOD


,Horisont,Modell,Träning,Test,Brier Score,ROC-AUC,Positiva utfall (%)
5,12M,Expanding Regression,2338,304,0.2840,0.7717,20.0658
4,12M,Baseline,2338,304,0.3079,0.5000,20.0658
6,12M,Rolling Regression,2338,304,0.3965,0.0919,20.0658
7,12M,Random Forest,2338,304,0.5258,0.1336,20.0658
1,6M,Expanding Regression,2523,489,0.2304,0.7512,50.9202
0,6M,Baseline,2523,489,0.2599,0.5000,50.9202
3,6M,Random Forest,2523,489,0.2880,0.5570,50.9202
2,6M,Rolling Regression,2523,489,0.3730,0.1862,50.9202


In [29]:

# STEG 13.2 – Modellernas Brier Score

import plotly.express as px

fig_brier13 = px.bar(
    df_test13,
    x="Modell",
    y="Brier Score",
    color="Modell",
    facet_col="Horisont",
    text_auto=".4f",
    title="ETH – Modelljämförelse på senare historisk testperiod",
    template="plotly_dark"
)

fig_brier13.update_layout(
    height=550,
    showlegend=False
)

fig_brier13.update_yaxes(
    title_text="Brier Score – lägre är bättre"
)

fig_brier13.show()


In [30]:

# STEG 13.3 – ROC-AUC för modelljämförelsen

fig_auc13 = px.bar(
    df_test13,
    x="Modell",
    y="ROC-AUC",
    color="Modell",
    facet_col="Horisont",
    text_auto=".3f",
    title="ETH – ROC-AUC under testperioden",
    template="plotly_dark"
)

fig_auc13.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="white"
)

fig_auc13.update_layout(
    height=550,
    showlegend=False
)

fig_auc13.update_yaxes(
    range=[0, 1]
)

fig_auc13.show()

print(
    "OBS: ROC-AUC visas bara när testperioden "
    "innehåller båda utfallsklasserna."
)


OBS: ROC-AUC visas bara när testperioden innehåller båda utfallsklasserna.


In [31]:

# STEG 13.5 – Utvärdera potentiella köpperioder

BUY_THRESHOLD = 0.60

zone_results13 = []

selected_models = {
    "6M": ("Rolling_Prob", "Rolling Regression"),
    "12M": ("Expanding_Prob", "Expanding Regression")
}

for horizon, (prob_col, model_name) in selected_models.items():

    subset = df_predictions13[
        df_predictions13["Horisont"] == horizon
    ].copy()

    subset["Köpklass"] = np.where(
        subset[prob_col] >= BUY_THRESHOLD,
        "Potentiell köpperiod",
        "Övriga datum"
    )

    for label, group in subset.groupby("Köpklass"):

        zone_results13.append({
            "Horisont": horizon,
            "Modell": model_name,
            "Kategori": label,
            "Observationer": len(group),
            "Medianavkastning (%)": (
                group["Actual_Return"].median()
            ),
            "Andel positiva (%)": (
                group["Target"].mean() * 100
            )
        })

df_zone13 = pd.DataFrame(zone_results13)

print("HISTORISK AVKASTNING – POTENTIELLA KÖPPERIODER")
display(df_zone13.round(2))

fig_zone13 = px.bar(
    df_zone13,
    x="Kategori",
    y="Medianavkastning (%)",
    color="Kategori",
    facet_col="Horisont",
    text_auto=".1f",
    title="ETH – Framtida medianavkastning per köpklass",
    template="plotly_dark"
)

fig_zone13.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

fig_zone13.update_layout(
    height=550,
    showlegend=False
)

fig_zone13.show()


HISTORISK AVKASTNING – POTENTIELLA KÖPPERIODER


,Horisont,Modell,Kategori,Observationer,Medianavkastning (%),Andel positiva (%)
0,6M,Rolling Regression,Potentiell köpperiod,329,-30.56,32.22
1,6M,Rolling Regression,Övriga datum,160,63.25,89.38
2,12M,Expanding Regression,Potentiell köpperiod,162,-15.95,36.42
3,12M,Expanding Regression,Övriga datum,142,-41.54,1.41



### Resultat och slutsats – Steg 13

#### Syfte

Vi undersökte om tidigare utvecklade modeller
kan identifiera gynnsamma köpperioder för Ethereum
under en senare historisk testperiod.

#### Metod

Fyra modeller jämfördes:

1. Baseline
2. Expanding Window Regression
3. Rolling Window Regression
4. Random Forest

Modellerna tränades på historiska observationer
med kända framtida utfall före 2025-01-01.

Prognoshorisonter:
- 6 månader: 180 dagar
- 12 månader: 365 dagar

#### Testperioder

**6 månader**
- Träningsobservationer: 2 523
- Testobservationer: 489
- Testperiod: 2025-01-01 till 2026-05-04
- Positiva framtida utfall: 50,92 %

**12 månader**
- Träningsobservationer: 2 338
- Testobservationer: 304
- Testperiod: 2025-01-01 till 2025-10-31
- Positiva framtida utfall: 20,07 %

#### Resultat – Brier Score

| Modell | 6M | 12M |
|---|---:|---:|
| Baseline | 0,2599 | 0,3079 |
| Expanding Regression | **0,2304** | **0,2840** |
| Rolling Regression | 0,3730 | 0,3965 |
| Random Forest | 0,2880 | 0,5258 |

Expanding Regression uppvisade lägst
prognosfel för båda horisonterna.

#### Resultat – ROC-AUC

| Modell | 6M | 12M |
|---|---:|---:|
| Baseline | 0,5000 | 0,5000 |
| Expanding Regression | **0,7512** | **0,7717** |
| Rolling Regression | 0,1862 | 0,0919 |
| Random Forest | 0,5570 | 0,1336 |

Expanding Regression hade högst ROC-AUC
för både 6 och 12 månader.

Det innebär att modellen hade bäst förmåga
att rangordna positiva och negativa utfall
under de undersökta perioderna.

#### Resultat – Potentiella köpperioder

För 6 månader utvärderades Rolling Regression,
som tidigare valts under modellvalideringen.

- Potentiella köpperioder: 329 observationer
- Medianavkastning: −30,56 %
- Andel positiva utfall: 32,22 %

Övriga datum hade en medianavkastning
på +63,25 %.

För 12 månader utvärderades Expanding Regression.

- Potentiella köpperioder: 162 observationer
- Medianavkastning: −15,95 %
- Andel positiva utfall: 36,42 %

Övriga datum hade en medianavkastning
på −41,54 %.

#### Analys

Resultaten visar att Expanding Regression
uppnådde bättre prognosprecision än de
övriga modellerna under testperioden.

Rolling Regression presterade däremot
sämre än referensmodellen.

Det visar att modellernas prestation
förändras mellan olika marknadsperioder.

För 6 månader gav de identifierade
köpperioderna negativ medianavkastning,
medan övriga datum hade positiv avkastning.

För 12 månader var köpperiodernas
medianavkastning högre än för övriga datum,
men fortfarande negativ.

Det innebär att en modell kan uppvisa
förmåga att skilja positiva och negativa
utfall utan att automatiskt identifiera
lönsamma investeringstillfällen.

#### Metodologiska begränsningar

- Testperioden har tidigare undersökts
  och är därför inte helt oberoende.
- Framtida avkastningsperioder överlappar.
- Sannolikhet för positiv avkastning
  innebär inte garanterad lönsamhet.
- Köpgränsen 60 % är en explorativ regel.
- Transaktionskostnader och slippage
  ingår inte i köpklassernas avkastningsjämförelse.
- Resultaten kan påverkas av marknadscykler
  och förändrade prisförhållanden.

#### Slutsats

Expanding Regression gav bäst resultat
enligt både Brier Score och ROC-AUC
under den senare historiska testperioden.

Detta ger visst stöd för att regressionsbaserade
zoner kan innehålla information om framtida
prisriktning för Ethereum.

Resultaten från köpperioderna är dock
inte tillräckligt konsekventa för att visa
att modellerna identifierar lönsamma köptillfällen.

Projektets forskningsfråga kan därför
ännu inte besvaras med ett entydigt ja.

Fortsatt arbete bör undersöka om
regressionsbaserade köpzoner ger bättre
riskjusterad avkastning än enkla
referensstrategier.



## Steg 14 – Köpzonsanalys med Expanding Regression

### Forskningsfråga

Kan logaritmisk regression identifiera historiskt
gynnsamma köpzoner för Ethereum?

### Syfte

Vi undersöker om Ethereums position i förhållande
till den historiskt beräknade regressionskanalen
har samband med framtida avkastning.

### Metod

Vi använder Expanding Window Regression från steg 4.

Tre zoner analyseras:

- Nedre zon: ETH-priset ligger under nedre bandet.
- Neutral zon: ETH-priset ligger mellan banden.
- Övre zon: ETH-priset ligger över övre bandet.

Framtida avkastning beräknas från nästa dags
pris och utvärderas efter:

- 180 dagar (6 månader)
- 365 dagar (12 månader)

Vi undersöker:

1. Medianavkastning
2. Andel positiva utfall
3. Andel förluster över 30 %
4. Avkastningens tionde percentil
5. Variation mellan kalenderår

### Viktig begränsning

En nedre regressionszon är inte automatiskt
en köpsignal.

Resultaten är en historisk jämförelse och inte
ett oberoende, färdigt handelssystem.


In [32]:

# STEG 14.1 – Köpzonsanalys för Ethereum

import numpy as np
import pandas as pd
from IPython.display import display

TEST_START = pd.Timestamp("2025-01-01", tz="UTC")

HORIZONS = {
    "6M": 180,
    "12M": 365
}

ZONE_ORDER = [
    "Nedre zon",
    "Neutral zon",
    "Övre zon"
]

# Utgå från steg 5
df_zone14 = df_returns.copy()

df_zone14["time"] = pd.to_datetime(
    df_zone14["time"], utc=True
)

df_zone14 = df_zone14.sort_values(
    "time"
).reset_index(drop=True)

# Endast giltiga historiska priszoner
df_zone14 = df_zone14[
    df_zone14["Price_Zone"].isin(ZONE_ORDER)
    & (df_zone14["time"] >= TEST_START)
].copy()

# Senast tillgängliga observationsdatum
last_available = df_eth["time"].max()

results14 = []
yearly14 = []
valid_rows = []

for horizon, days in HORIZONS.items():

    return_col = f"Return_{horizon}"

    subset = df_zone14.copy()

    # Köp nästa dag efter signal
    subset["Entry_Date"] = (
        subset["time"] + pd.Timedelta(days=1)
    )

    subset["Exit_Date"] = (
        subset["Entry_Date"] + pd.Timedelta(days=days)
    )

    # Endast kända framtida utfall
    subset = subset[
        subset[return_col].notna()
        & (subset["Exit_Date"] <= last_available)
    ].copy()

    subset["Horisont"] = horizon
    subset["Avkastning"] = subset[return_col]
    subset["År"] = subset["time"].dt.year

    valid_rows.append(subset)

    print(f"\n===== {horizon} =====")
    print("Observationer:", len(subset))

    if subset.empty:
        continue

    print("Första signaldatum:", subset["time"].min())
    print("Sista signaldatum:", subset["time"].max())

    # Referens: samtliga giltiga köpdatum
    baseline_median = subset["Avkastning"].median()

    for zone in ZONE_ORDER:

        zone_data = subset[
            subset["Price_Zone"] == zone
        ]

        values = zone_data["Avkastning"].dropna()

        if values.empty:
            continue

        results14.append({
            "Horisont": horizon,
            "Priszon": zone,
            "Observationer": len(values),
            "Medianavkastning (%)": values.median(),
            "Medelavkastning (%)": values.mean(),
            "Positiva utfall (%)": (
                values.gt(0).mean() * 100
            ),
            "Förlust över 30 % (%)": (
                values.lt(-30).mean() * 100
            ),
            "10:e percentilen (%)": values.quantile(0.10),
            "Skillnad mot referens (pp)": (
                values.median() - baseline_median
            )
        })

    # Årsvis sammanställning
    for (year, zone), group in subset.groupby(
        ["År", "Price_Zone"]
    ):

        yearly14.append({
            "Horisont": horizon,
            "År": year,
            "Priszon": zone,
            "Observationer": len(group),
            "Medianavkastning (%)": (
                group["Avkastning"].median()
            ),
            "Positiva utfall (%)": (
                group["Avkastning"].gt(0).mean() * 100
            )
        })

# Samla tabeller
df_zone14_valid = pd.concat(
    valid_rows,
    ignore_index=True
)

df_zone14_results = pd.DataFrame(results14)
df_zone14_yearly = pd.DataFrame(yearly14)

print("\nTABELL 1 – AVKASTNING OCH RISK PER ZON")
display(df_zone14_results.round(2))

print("\nTABELL 2 – RESULTAT PER ÅR")
display(df_zone14_yearly.round(2))

print("\nTABELL 3 – SENASTE GILTIGA OBSERVATIONER")
display(
    df_zone14_valid[
        [
            "time",
            "Horisont",
            "Price_Zone",
            "Entry_Price",
            "Avkastning"
        ]
    ].tail(10).round(2)
)



===== 6M =====
Observationer: 489
Första signaldatum: 2025-01-01 00:00:00+00:00
Sista signaldatum: 2026-05-04 00:00:00+00:00

===== 12M =====
Observationer: 304
Första signaldatum: 2025-01-01 00:00:00+00:00
Sista signaldatum: 2025-10-31 00:00:00+00:00

TABELL 1 – AVKASTNING OCH RISK PER ZON


,Horisont,Priszon,Observationer,Medianavkastning (%),Medelavkastning (%),Positiva utfall (%),Förlust över 30 % (%),10:e percentilen (%),Skillnad mot referens (pp)
0,6M,Nedre zon,337,19.05,29.12,69.14,19.88,-42.36,16.72
1,6M,Neutral zon,152,-41.09,-33.84,10.53,65.79,-55.13,-43.42
2,12M,Nedre zon,162,-15.95,-6.72,36.42,22.84,-34.88,13.06
3,12M,Neutral zon,142,-41.54,-36.24,1.41,77.46,-49.86,-12.53



TABELL 2 – RESULTAT PER ÅR


,Horisont,År,Priszon,Observationer,Medianavkastning (%),Positiva utfall (%)
0,6M,2025,Nedre zon,213,27.81,76.06
1,6M,2025,Neutral zon,152,-41.09,10.53
2,6M,2026,Nedre zon,124,3.46,57.26
3,12M,2025,Nedre zon,162,-15.95,36.42
4,12M,2025,Neutral zon,142,-41.54,1.41



TABELL 3 – SENASTE GILTIGA OBSERVATIONER


C:\Users\milen\AppData\Local\Temp\ipykernel_10760\3199295536.py:155: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,Horisont,Price_Zone,Entry_Price,Avkastning
783,2025-10-22 00:00:00+00:00,12M,Neutral zon,3856.10,-38.32
784,2025-10-23 00:00:00+00:00,12M,Neutral zon,3932.93,-40.20
785,2025-10-24 00:00:00+00:00,12M,Neutral zon,3955.02,-41.19
786,2025-10-25 00:00:00+00:00,12M,Neutral zon,4166.26,-44.40
787,2025-10-26 00:00:00+00:00,12M,Neutral zon,4118.91,-43.01
788,2025-10-27 00:00:00+00:00,12M,Neutral zon,3984.37,-40.84
789,2025-10-28 00:00:00+00:00,12M,Neutral zon,3902.84,-42.20
790,2025-10-29 00:00:00+00:00,12M,Neutral zon,3791.08,-40.91
791,2025-10-30 00:00:00+00:00,12M,Neutral zon,3851.15,-41.52
792,2025-10-31 00:00:00+00:00,12M,Neutral zon,3871.70,-41.37


In [33]:

# STEG 14.2 – ETH-pris med historiska köpzoner

import plotly.graph_objects as go

fig_zone_price = go.Figure()

# Övre regressionsband
fig_zone_price.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["Upper_Band"],
    name="Övre regressionsband",
    mode="lines",
    line=dict(color="#2ECC71", width=1)
))

# Nedre band
fig_zone_price.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["Lower_Band"],
    name="Nedre regressionsband",
    mode="lines",
    fill="tonexty",
    fillcolor="rgba(46,204,113,0.12)",
    line=dict(color="#2ECC71", width=1)
))

# Regressionslinje
fig_zone_price.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["Regression_Trend"],
    name="Expanding Regression",
    line=dict(color="#F39C12", width=2)
))

# ETH-pris
fig_zone_price.add_trace(go.Scatter(
    x=df_exp["time"],
    y=df_exp["ETH_Price"],
    name="ETH-pris",
    mode="lines",
    line=dict(color="#5DADE2", width=1.5)
))

# Markera zoner
zone_colors = {
    "Nedre zon": "#FF4B4B",
    "Neutral zon": "#F39C12",
    "Övre zon": "#2ECC71"
}

# Markera en punkt var sjunde observation
# för att diagrammet ska bli lättläst
for zone in ZONE_ORDER:

    points = df_exp[
        df_exp["Price_Zone"] == zone
    ].iloc[::7]

    fig_zone_price.add_trace(go.Scatter(
        x=points["time"],
        y=points["ETH_Price"],
        mode="markers",
        name=zone,
        marker=dict(
            color=zone_colors[zone],
            size=5,
            opacity=0.8
        )
    ))

fig_zone_price.update_layout(
    title="Ethereum – Historiska priszoner och regressionsband",
    xaxis_title="Datum",
    yaxis_title="ETH-pris (USD, logaritmisk skala)",
    yaxis_type="log",
    template="plotly_dark",
    height=700,
    hovermode="x unified"
)

fig_zone_price.update_xaxes(
    rangeslider_visible=True
)

fig_zone_price.show()


In [34]:

# STEG 14.3 – Avkastning per priszon

import plotly.express as px

fig_zone_return = px.bar(
    df_zone14_results,
    x="Priszon",
    y="Medianavkastning (%)",
    color="Horisont",
    barmode="group",
    text_auto=".1f",
    category_orders={"Priszon": ZONE_ORDER},
    title="ETH – Framtida medianavkastning per köpzon",
    template="plotly_dark"
)

fig_zone_return.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

fig_zone_return.update_layout(
    height=550,
    xaxis_title="Regressionszon",
    yaxis_title="Medianavkastning (%)"
)

fig_zone_return.show()


In [35]:

# STEG 14.4 – Risk per köpzon

fig_zone_risk = px.bar(
    df_zone14_results,
    x="Priszon",
    y="Förlust över 30 % (%)",
    color="Horisont",
    barmode="group",
    text_auto=".1f",
    category_orders={"Priszon": ZONE_ORDER},
    title="ETH – Andel historiska förluster över 30 %",
    template="plotly_dark"
)

fig_zone_risk.update_layout(
    height=550,
    xaxis_title="Regressionszon",
    yaxis_title="Andel utfall med avkastning under −30 % (%)",
    yaxis_range=[0, 100]
)

fig_zone_risk.show()


In [36]:

# STEG 14.5 – Köpzonernas utveckling per år

fig_zone_yearly = px.bar(
    df_zone14_yearly,
    x="År",
    y="Medianavkastning (%)",
    color="Priszon",
    barmode="group",
    facet_row="Horisont",
    text_auto=".1f",
    title="ETH – Köpzonsresultat per år",
    template="plotly_dark",
    category_orders={"Priszon": ZONE_ORDER}
)

fig_zone_yearly.update_xaxes(type="category")

fig_zone_yearly.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

fig_zone_yearly.update_layout(
    height=750,
    yaxis_title="Medianavkastning (%)"
)

fig_zone_yearly.show()



### Resultat och slutsats – Steg 14

#### Syfte

Vi undersökte om Expanding Window Regression
kan identifiera historiskt gynnsamma köpzoner
för Ethereum under perioden 2025–2026.

#### Metod

Vi använde tre regressionsbaserade priszoner:

- Nedre zon
- Neutral zon
- Övre zon

Framtida avkastning beräknades från nästa dags
ETH-pris efter 180 respektive 365 dagar.

Endast observationer med kända framtida
utfall inkluderades.

#### Resultat – 6 månader

| Mått | Nedre zon | Neutral zon |
|---|---:|---:|
| Observationer | 337 | 152 |
| Medianavkastning | +19,05 % | −41,09 % |
| Medelavkastning | +29,12 % | −33,84 % |
| Positiva utfall | 69,14 % | 10,53 % |
| Förlust över 30 % | 19,88 % | 65,79 % |
| 10:e percentilen | −42,36 % | −55,13 % |

Den nedre regressionszonen gav 60,14
procentenheter högre medianavkastning
än den neutrala zonen.

Andelen positiva utfall var betydligt
högre i den nedre zonen.

Risken för en förlust större än 30 %
var också lägre i den nedre zonen.

#### Resultat – 12 månader

| Mått | Nedre zon | Neutral zon |
|---|---:|---:|
| Observationer | 162 | 142 |
| Medianavkastning | −15,95 % | −41,54 % |
| Medelavkastning | −6,72 % | −36,24 % |
| Positiva utfall | 36,42 % | 1,41 % |
| Förlust över 30 % | 22,84 % | 77,46 % |
| 10:e percentilen | −34,88 % | −49,86 % |

Den nedre zonen hade 25,59 procentenheter
högre medianavkastning än den neutrala zonen.

Båda zonerna uppvisade dock negativ
medianavkastning efter 12 månader.

Det innebär att den nedre zonen var
relativt mer gynnsam, men inte generellt
lönsam under den undersökta perioden.

#### Resultat per kalenderår

Under 2025 hade nedre zonen en
medianavkastning på +27,81 % efter 6 månader.

Under den utvärderingsbara delen av 2026
var motsvarande medianavkastning +3,46 %.

För 12 månader fanns endast fullständiga
framtida utfall för signaldatum under 2025.

#### Analys

Resultaten visar ett historiskt samband
mellan regressionszon och framtida avkastning.

Den nedre zonen hade högre medianavkastning,
fler positiva utfall och färre stora förluster
än den neutrala zonen.

Detta ger preliminärt stöd för att
regressionszoner kan användas för att
identifiera relativt gynnsamma köpperioder.

Samtidigt är resultaten inte tillräckliga
för att fastställa tillförlitlig
framtida prognosförmåga.

#### Metodologiska begränsningar

- Framtida avkastningsperioder överlappar.
- Observationerna är inte oberoende.
- Testperioden har tidigare undersökts.
- Övre zonen saknar observationer under perioden.
- Resultaten är inte ett fullständigt backtest.
- Transaktionskostnader och slippage ingår inte.
- Högre relativ avkastning innebär inte
  nödvändigtvis positiv absolut avkastning.
- Marknadsförhållanden kan förändras över tid.

#### Slutsats

Expanding Window Regression identifierade
historiska priszoner med tydliga skillnader
i framtida avkastning och risk.

Den nedre zonen uppvisade bättre resultat
än den neutrala zonen efter både
6 och 12 månader.

För 6 månader var medianavkastningen
positiv i den nedre zonen.

För 12 månader var medianavkastningen
negativ även i den nedre zonen.

Resultaten ger därmed visst stöd för
projektets hypotes om historiskt
gynnsamma ackumulationszoner.

Vi kan däremot ännu inte fastställa
att modellen är tillförlitlig för
framtida investeringsbeslut.

Ytterligare oberoende validering och
riskjusterad utvärdering behövs.



# Steg 15 – Slutlig modellrapport

## Ethereum Accumulation Zones

### Forskningsfråga

Kan maskininlärning och logaritmisk regression
identifiera historiskt gynnsamma köpzoner för
Ethereum med hjälp av ETH- och BTC-pristidsserier?

### Syfte

Vi sammanfattar resultaten från tidigare analyser
och undersöker om regressionsbaserade modeller
eller Random Forest kan bidra till att identifiera
perioder med relativt gynnsam framtida avkastning.

### Genomförda analyser

1. Datainsamling och datakvalitet
2. Explorativ dataanalys (EDA)
3. Logaritmisk regression
4. Expanding Window Regression
5. Framtida avkastning per priszon
6. Årsvis analys
7. Tidsbaserad utvärdering
8. Historisk backtesting
9. Strategins stabilitet
10. Expanding Window Cross-Validation
11. Tolvmånadersprojektion
12. Jämförelse med Random Forest
13. Senare historisk testperiod
14. Analys av köpzoner och risk

### Utvärdering

Vi sammanfattar modellernas klassificeringsförmåga
med Brier Score och ROC-AUC.

Vi bedömer även ekonomiska resultat med
avkastning, förluster och maximal drawdown.

Detta är en explorativ historisk utvärdering.


In [37]:

# STEG 15.1 – Slutlig sammanställning

import pandas as pd
import numpy as np
from IPython.display import display

# Kontrollera att tidigare resultat finns
required = [
    "df_model_summary",
    "df_test13",
    "df_zone14_results",
    "summary_bt"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise ValueError(
        "Kör först tidigare celler. Saknas: "
        + ", ".join(missing)
    )

# Resultat från steg 12
validation = df_model_summary[
    ["Horisont", "Modell", "Brier Score"]
].copy()

validation = validation.rename(columns={
    "Brier Score": "Validering_Brier"
})

# Resultat från steg 13
test = df_test13[
    ["Horisont", "Modell", "Brier Score", "ROC-AUC"]
].copy()

test = test.rename(columns={
    "Brier Score": "Test_Brier",
    "ROC-AUC": "Test_ROC_AUC"
})

# Slå samman
df_final_models = validation.merge(
    test,
    on=["Horisont", "Modell"],
    how="inner",
    validate="one_to_one"
)

# Sortera
df_final_models = df_final_models.sort_values(
    ["Horisont", "Test_Brier"]
)

print("SLUTLIG MODELLJÄMFÖRELSE")
display(df_final_models.round(4))

# Köpzonernas resultat
print("\nKÖPZONERNAS AVKASTNING OCH RISK")

display(
    df_zone14_results[
        [
            "Horisont",
            "Priszon",
            "Observationer",
            "Medianavkastning (%)",
            "Positiva utfall (%)",
            "Förlust över 30 % (%)"
        ]
    ].round(2)
)

# Backtesting
print("\nBACKTESTING MOT BUY & HOLD")

display(summary_bt.round(2))


SLUTLIG MODELLJÄMFÖRELSE


,Horisont,Modell,Validering_Brier,Test_Brier,Test_ROC_AUC
1,12M,Expanding Regression,0.2292,0.2840,0.7717
0,12M,Baseline,0.2427,0.3079,0.5000
2,12M,Rolling Regression,0.2873,0.3965,0.0919
3,12M,Random Forest,0.2426,0.5258,0.1336
5,6M,Expanding Regression,0.2867,0.2304,0.7512
4,6M,Baseline,0.2617,0.2599,0.5000
7,6M,Random Forest,0.3052,0.2880,0.5570
6,6M,Rolling Regression,0.2374,0.3730,0.1862



KÖPZONERNAS AVKASTNING OCH RISK


,Horisont,Priszon,Observationer,Medianavkastning (%),Positiva utfall (%),Förlust över 30 % (%)
0,6M,Nedre zon,337,19.05,69.14,19.88
1,6M,Neutral zon,152,-41.09,10.53,65.79
2,12M,Nedre zon,162,-15.95,36.42,22.84
3,12M,Neutral zon,142,-41.54,1.41,77.46



BACKTESTING MOT BUY & HOLD


,Strategi,Slutkapital USD,Total avkastning (%),Max Drawdown (%)
0,Ackumulationszon,37457.03,274.57,-63.62
1,Buy & Hold,163127.03,1531.27,-79.36


In [38]:

# STEG 15.2 – Validering jämfört med senare test

import plotly.express as px

plot_models = df_final_models.melt(
    id_vars=["Horisont", "Modell"],
    value_vars=["Validering_Brier", "Test_Brier"],
    var_name="Period",
    value_name="Brier Score"
)

plot_models["Period"] = plot_models["Period"].replace({
    "Validering_Brier": "Validering 2021–2024",
    "Test_Brier": "Senare test 2025–2026"
})

fig_final_models = px.bar(
    plot_models,
    x="Modell",
    y="Brier Score",
    color="Period",
    barmode="group",
    facet_col="Horisont",
    text_auto=".3f",
    title="Ethereum – Modellernas prognosprecision",
    template="plotly_dark"
)

fig_final_models.update_layout(
    height=600,
    xaxis_title="Modell"
)

fig_final_models.update_yaxes(
    title_text="Brier Score (lägre är bättre)"
)

fig_final_models.show()


In [39]:

# STEG 15.3 – Slutliga köpzonsdiagram

import plotly.express as px

# Framtida medianavkastning
fig_returns = px.bar(
    df_zone14_results,
    x="Priszon",
    y="Medianavkastning (%)",
    color="Horisont",
    barmode="group",
    text_auto=".1f",
    title="Ethereum – Medianavkastning per regressionszon",
    template="plotly_dark",
    category_orders={
        "Priszon": [
            "Nedre zon",
            "Neutral zon",
            "Övre zon"
        ]
    }
)

fig_returns.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

fig_returns.update_layout(
    height=550,
    yaxis_title="Medianavkastning (%)"
)

fig_returns.show()

# Andel historiska stora förluster
fig_losses = px.bar(
    df_zone14_results,
    x="Priszon",
    y="Förlust över 30 % (%)",
    color="Horisont",
    barmode="group",
    text_auto=".1f",
    title="Ethereum – Risk för förlust över 30 %",
    template="plotly_dark"
)

fig_losses.update_layout(
    height=550,
    yaxis_range=[0, 100],
    yaxis_title="Andel observationer (%)"
)

fig_losses.show()


In [40]:

# STEG 15.4 – Sammanfattande backtesting

import plotly.express as px

final_bt = summary_bt.copy()

# Slutkapital
fig_capital_final = px.bar(
    final_bt,
    x="Strategi",
    y="Slutkapital USD",
    color="Strategi",
    text_auto=",.0f",
    title="Ethereum – Slutkapital från 10 000 USD",
    template="plotly_dark"
)

fig_capital_final.update_layout(
    height=500,
    showlegend=False
)

fig_capital_final.show()

# Maximal drawdown
fig_drawdown_final = px.bar(
    final_bt,
    x="Strategi",
    y="Max Drawdown (%)",
    color="Strategi",
    text_auto=".1f",
    title="Ethereum – Maximal historisk drawdown",
    template="plotly_dark"
)

fig_drawdown_final.update_layout(
    height=500,
    showlegend=False
)

fig_drawdown_final.show()


In [41]:

# STEG 15.5 – Exportera sammanfattning till Streamlit

from pathlib import Path

# Mapp för exporterade resultat
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

exports = {
    "eth_final_model_comparison.csv": df_final_models,
    "eth_final_buy_zones.csv": df_zone14_results,
    "eth_final_backtesting.csv": summary_bt,
    "eth_final_yearly_zones.csv": df_zone14_yearly
}

for filename, dataframe in exports.items():

    filepath = output_dir / filename

    dataframe.to_csv(
        filepath,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Exporterad: {filepath}")

print("\nAlla sammanfattande resultat har exporterats.")


Exporterad: data\eth_final_model_comparison.csv
Exporterad: data\eth_final_buy_zones.csv
Exporterad: data\eth_final_backtesting.csv
Exporterad: data\eth_final_yearly_zones.csv

Alla sammanfattande resultat har exporterats.


In [42]:

# STEG 16.1 – Exportera dagliga analysdata till Streamlit

from pathlib import Path
import pandas as pd

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

# Dagliga ETH-priser, regressionsband och priszoner
eth_streamlit = df_exp[
    [
        "time",
        "ETH_Price",
        "Regression_Trend",
        "Upper_Band",
        "Lower_Band",
        "Zone_Score",
        "Price_Zone"
    ]
].copy()

eth_streamlit.to_csv(
    output_dir / "eth_daily_accumulation_zones.csv",
    index=False,
    encoding="utf-8-sig"
)

# Historisk kapitalutveckling
backtest_streamlit = bt[
    [
        "time",
        "ETH_Price",
        "Strategy_Value",
        "BuyHold_Value",
        "Strategy_Drawdown",
        "BuyHold_Drawdown"
    ]
].copy()

backtest_streamlit.to_csv(
    output_dir / "eth_backtest_history.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Exporten är klar!")
print("Mapp:", output_dir.resolve())

for filename in [
    "eth_daily_accumulation_zones.csv",
    "eth_backtest_history.csv"
]:
    print(filename)


Exporten är klar!
Mapp: C:\Users\milen\source\repos\Statistisk_programmering_ML\Ethereum_Bitcoin_ML\data
eth_daily_accumulation_zones.csv
eth_backtest_history.csv
